# AnemiaLens — training model skrining anemia dari foto konjungtiva

Jalankan sel **berurutan dari atas ke bawah**. Setiap langkah ada penjelasan dan perkiraan waktunya.

**Sebelum mulai:** menu *Runtime → Change runtime type → T4 GPU → Save*.

Alur yang dibangun:
1. **Segmentasi** (U-Net) memotong area konjungtiva palpebral dari foto mata.
2. **Empat model** menilai potongan itu:
   - fitur warna klasik (CIELAB a\*, erythema index, high hue ratio) → regresi logistik **dan** Random Forest,
   - DINOv2 ViT-S/14 sebagai *linear probe*,
   - EfficientNet-B0 multi-task (klasifikasi anemia + regresi nilai Hb).
3. **Ensemble** menghasilkan tingkat risiko (rendah/sedang/tinggi) dan perkiraan Hb.
4. **Validasi silang 5-fold per pasien** + uji *leave-one-source-out*. Angka yang dilaporkan jujur, tanpa kebocoran data.

## Langkah 1 — Cek GPU
Harus muncul **Tesla T4** (atau GPU lain). Kalau error, ganti runtime ke GPU dulu.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Langkah 2 — Pasang library (±1 menit)

In [ ]:
!pip install -q timm==1.0.* segmentation-models-pytorch==0.5.* gradio gdown openpyxl tabulate

## Langkah 3 — Tulis kode AnemiaLens ke Colab
Sel ini membuat folder `/content/anemialens` berisi semua script. Tidak perlu diubah.

In [ ]:
import os, json
os.makedirs('/content/anemialens', exist_ok=True)
FILES = {"__init__.py": "", "common.py": "\"\"\"AnemiaLens - utilitas bersama: konfigurasi, I/O gambar, ambang anemia WHO,\npemeriksaan kualitas foto, pemotongan ROI konjungtiva, dan fitur warna.\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom dataclasses import dataclass, asdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nIMG_EXT = {\".jpg\", \".jpeg\", \".png\", \".bmp\", \".tif\", \".tiff\", \".webp\"}\n# Kata kunci nama file yang menandakan mask segmentasi, bukan foto asli\nMASK_KEYS = (\"mask\", \"palpebral\", \"palplebral\", \"papebral\", \"palpebrale\", \"forniceal\", \"fornice\", \"segment\", \"_seg\", \"label\")\n\n\n@dataclass\nclass Config:\n    work_dir: str = \"/content/work\"          # tempat semua keluaran\n    seed: int = 42\n    n_folds: int = 5\n    img_size: int = 224                      # input classifier\n    seg_size: int = 384                      # input segmentasi\n    cnn_backbone: str = \"efficientnet_b0\"    # alternatif: convnext_tiny.fb_in22k_ft_in1k\n    dino_backbone: str = \"vit_small_patch14_dinov2.lvd142m\"\n    cnn_epochs: int = 20\n    seg_epochs: int = 40\n    batch_size: int = 32\n    lr: float = 3e-4\n    weight_decay: float = 1e-4\n    hb_loss_weight: float = 0.3\n    target_sensitivity: float = 0.85         # ambang dipilih agar sensitivitas >= ini\n    num_workers: int = 2\n    pretrained: bool = True                  # False hanya untuk uji offline\n\n    def save(self, path):\n        Path(path).write_text(json.dumps(asdict(self), indent=2))\n\n    @classmethod\n    def load(cls, path):\n        return cls(**json.loads(Path(path).read_text()))\n\n\ndef seed_everything(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef is_image(p: Path) -> bool:\n    return p.suffix.lower() in IMG_EXT\n\n\ndef looks_like_mask(p: Path) -> bool:\n    s = p.stem.lower()\n    return any(k in s for k in MASK_KEYS)\n\n\ndef read_rgb(path) -> np.ndarray:\n    \"\"\"Baca RGB dengan rotasi EXIF diterapkan (foto HP sering disimpan miring + tag rotasi;\n    mask dataset dibuat pada orientasi yang sudah diputar). Piksel transparan (PNG RGBA)\n    dijadikan hitam = latar ROI.\"\"\"\n    buf = np.fromfile(str(path), dtype=np.uint8)\n    raw = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)\n    if raw is None:\n        raise ValueError(f\"Gagal membaca gambar: {path}\")\n    if raw.ndim == 3 and raw.shape[2] == 4:\n        if raw.dtype != np.uint8:\n            raw = cv2.normalize(raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)\n        a = raw[:, :, 3:4].astype(np.float32) / 255.0\n        img = (raw[:, :, :3].astype(np.float32) * a).astype(np.uint8)\n    else:\n        img = cv2.imdecode(buf, cv2.IMREAD_COLOR)      # IMREAD_COLOR menerapkan orientasi EXIF\n    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n\n\ndef _fill_holes(m):\n    cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    out = np.zeros_like(m, dtype=np.uint8)\n    cv2.drawContours(out, cnts, -1, 1, thickness=-1)\n    return out\n\n\ndef read_mask(path, shape=None) -> np.ndarray:\n    \"\"\"Baca mask dalam berbagai format yang ada di dataset:\n    - mask biner hitam/putih (atau abu-abu),\n    - PNG transparan: area objek = alfa > 0,\n    - potongan konjungtiva berwarna di atas latar polos (putih/hitam), mis. Eyes-defy Italia 001-031.\n    Latar ditentukan dari warna median piksel tepi gambar.\"\"\"\n    m = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_UNCHANGED)\n    if m is None:\n        raise ValueError(f\"Gagal membaca mask: {path}\")\n    if m.dtype != np.uint8:\n        m = cv2.normalize(m, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)\n    if m.ndim == 3 and m.shape[2] == 4:\n        alpha = m[:, :, 3]\n        frac = (alpha > 127).mean()\n        if 0.002 < frac < 0.98:\n            out = (alpha > 127).astype(np.uint8)\n            m = None\n        else:\n            m = m[:, :, :3]\n    if m is not None:\n        if m.ndim == 2:\n            m = m[:, :, None]\n        edge = np.concatenate([m[0], m[-1], m[:, 0], m[:, -1]]).astype(np.int16)\n        bg = np.median(edge, axis=0)\n        diff = np.abs(m.astype(np.int16) - bg).max(axis=2)\n        out = (diff > 40).astype(np.uint8)\n        out = cv2.morphologyEx(out, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n        out = _fill_holes(out)\n    if shape is not None and out.shape[:2] != tuple(shape[:2]):\n        out = cv2.resize(out, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)\n    return out\n\n\n# ---------------------------------------------------------------- ambang WHO\ndef anemia_threshold(age=None, sex=None) -> float:\n    \"\"\"Ambang Hb (g/dL) anemia menurut WHO.\n    6-59 bulan: 11 | 5-11 th: 11.5 | 12-14 th: 12 | >=15 th: perempuan 12, laki-laki 13.\n    Usia tidak diketahui -> 12 (paling umum untuk remaja/dewasa).\"\"\"\n    try:\n        age = float(age)\n    except (TypeError, ValueError):\n        age = None\n    s = str(sex).strip().lower() if sex is not None else \"\"\n    male = s in {\"m\", \"male\", \"l\", \"laki-laki\", \"laki\", \"pria\", \"1\", \"man\"}\n    if age is not None:\n        if age < 5:\n            return 11.0\n        if age < 12:\n            return 11.5\n        if age < 15:\n            return 12.0\n    return 13.0 if male else 12.0\n\n\n# ---------------------------------------------------------- kualitas foto\ndef quality_check(rgb: np.ndarray, mask: np.ndarray | None = None, ref: dict | None = None) -> dict:\n    \"\"\"Mengembalikan metrik kualitas + daftar masalah. Dipakai saat training\n    (untuk laporan) dan saat inferensi (untuk menolak foto buruk).\"\"\"\n    h, w = rgb.shape[:2]\n    scale = 512 / max(h, w)\n    small = cv2.resize(rgb, (int(w * scale), int(h * scale))) if scale < 1 else rgb\n    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)\n    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)\n    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())\n    bright = float(hsv[:, :, 2].mean())\n    ref = ref or {}\n    issues = []\n    if blur < min(40.0, ref.get(\"blur_min\", 40.0)):\n        issues.append(\"buram\")\n    if bright < min(50.0, ref.get(\"bright_min\", 50.0)):\n        issues.append(\"terlalu_gelap\")\n    if bright > max(225.0, ref.get(\"bright_max\", 225.0)):\n        issues.append(\"terlalu_terang\")\n    glare = 0.0\n    area = None\n    if mask is not None and mask.any():\n        m = cv2.resize(mask, (small.shape[1], small.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)\n        v, s = hsv[:, :, 2][m], hsv[:, :, 1][m]\n        glare = float(((v > 245) & (s < 40)).mean())\n        area = float(mask.mean())\n        if glare > max(0.08, ref.get(\"glare_max\", 0.08)):\n            issues.append(\"pantulan_cahaya\")\n        # terlalu kecil = terlalu sedikit piksel untuk menilai warna (bukan sekadar persentase,\n        # agar konjungtiva yang dilingkari pada foto besar tetap diterima)\n        if int(mask.sum()) < 250 or area < 0.002:\n            issues.append(\"konjungtiva_terlalu_kecil\")\n    elif mask is not None:\n        issues.append(\"konjungtiva_tidak_terdeteksi\")\n    return {\"blur\": blur, \"brightness\": bright, \"glare\": glare, \"mask_area\": area, \"issues\": issues}\n\n\n# ------------------------------------------------------------- crop ROI\ndef crop_roi(rgb: np.ndarray, mask: np.ndarray | None, pad: float = 0.08, zero_bg: bool = True) -> np.ndarray:\n    \"\"\"Potong kotak pembatas konjungtiva. Piksel di luar mask dihitamkan supaya\n    model fokus pada warna konjungtiva, bukan kulit/sklera/latar.\"\"\"\n    if mask is None or not mask.any():\n        return rgb.copy()\n    ys, xs = np.where(mask > 0)\n    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()\n    ph, pw = int((y1 - y0) * pad), int((x1 - x0) * pad)\n    y0, x0 = max(0, y0 - ph), max(0, x0 - pw)\n    y1, x1 = min(rgb.shape[0], y1 + ph + 1), min(rgb.shape[1], x1 + pw + 1)\n    out = rgb[y0:y1, x0:x1].copy()\n    if zero_bg:\n        out[mask[y0:y1, x0:x1] == 0] = 0\n    return out\n\n\ndef letterbox(img: np.ndarray, size: int) -> np.ndarray:\n    h, w = img.shape[:2]\n    s = size / max(h, w)\n    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))\n    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)\n    out = np.zeros((size, size, 3), dtype=img.dtype)\n    y, x = (size - nh) // 2, (size - nw) // 2\n    out[y:y + nh, x:x + nw] = r\n    return out\n\n\n# --------------------------------------------------------- white balance\ndef white_balance_from_patch(rgb: np.ndarray, patch_box) -> np.ndarray:\n    \"\"\"Koreksi warna memakai kartu referensi putih/abu yang ikut difoto.\n    patch_box = (x0, y0, x1, y1) area kartu. Menyamakan rata-rata kartu ke abu netral.\"\"\"\n    x0, y0, x1, y1 = [int(v) for v in patch_box]\n    patch = rgb[y0:y1, x0:x1].reshape(-1, 3).astype(np.float32)\n    mean = np.median(patch, axis=0) + 1e-6\n    gain = np.clip(mean.mean() / mean, 0.8, 1.25)   # batasi koreksi agar tidak ekstrem\n    return np.clip(rgb.astype(np.float32) * gain, 0, 255).astype(np.uint8)\n\n\n# ------------------------------------------------------------ fitur warna\nCOLOR_FEATURE_NAMES = [\n    \"R_mean\", \"G_mean\", \"B_mean\", \"R_med\", \"G_med\", \"B_med\",\n    \"L_mean\", \"a_mean\", \"b_mean\", \"a_med\", \"a_p25\", \"a_p75\",\n    \"H_mean\", \"S_mean\", \"V_mean\",\n    \"erythema_index\", \"r_chroma\", \"g_chroma\", \"RG_ratio\", \"RB_ratio\",\n    \"high_hue_ratio\", \"entropy_R\", \"a_std\", \"L_std\",\n]\n\n\ndef color_features(roi_rgb: np.ndarray) -> np.ndarray:\n    \"\"\"Fitur warna klasik dari literatur pucat konjungtiva: CIELAB a*,\n    erythema index, rasio kromatik, high hue ratio (HHR), dan entropi.\n    Hanya piksel non-hitam (di dalam mask) yang dihitung.\"\"\"\n    px = roi_rgb.reshape(-1, 3)\n    valid = px.sum(axis=1) > 15\n    if valid.sum() < 50:\n        valid = np.ones(len(px), dtype=bool)\n    sel = roi_rgb.reshape(-1, 1, 3)[valid]\n    rgb = sel.reshape(-1, 3).astype(np.float32)\n    lab = cv2.cvtColor(sel.astype(np.uint8), cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)\n    lab[:, 0] *= 100 / 255\n    lab[:, 1:] -= 128\n    hsv = cv2.cvtColor(sel.astype(np.uint8), cv2.COLOR_RGB2HSV).reshape(-1, 3).astype(np.float32)\n    R, G, B = rgb[:, 0], rgb[:, 1], rgb[:, 2]\n    s = R + G + B + 1e-6\n    ei = np.log10(R + 1) - np.log10(G + 1)\n    hue = hsv[:, 0] * 2.0                     # OpenCV H 0-180 -> derajat\n    hhr = float(((hue < 15) | (hue > 345)).mean())\n    hist = np.histogram(R, bins=32, range=(0, 255))[0].astype(np.float64)\n    p = hist / (hist.sum() + 1e-9)\n    ent = float(-(p[p > 0] * np.log2(p[p > 0])).sum())\n    a = lab[:, 1]\n    feats = [\n        R.mean(), G.mean(), B.mean(), np.median(R), np.median(G), np.median(B),\n        lab[:, 0].mean(), a.mean(), lab[:, 2].mean(), np.median(a), np.percentile(a, 25), np.percentile(a, 75),\n        hue.mean(), hsv[:, 1].mean(), hsv[:, 2].mean(),\n        ei.mean(), (R / s).mean(), (G / s).mean(), (R / (G + 1)).mean(), (R / (B + 1)).mean(),\n        hhr, ent, a.std(), lab[:, 0].std(),\n    ]\n    return np.asarray(feats, dtype=np.float32)\n\n\ndef risk_tier(prob: float, thr_sens: float, thr_spec: float) -> str:\n    \"\"\"3 tingkat risiko dari probabilitas ensemble.\n    tinggi: >= ambang spesifisitas tinggi | sedang: >= ambang sensitivitas | rendah: lainnya.\"\"\"\n    if prob >= thr_spec:\n        return \"tinggi\"\n    if prob >= thr_sens:\n        return \"sedang\"\n    return \"rendah\"\n", "inspect_data.py": "\"\"\"Langkah 3: lihat isi dataset sebelum training.\nPakai:  python anemialens/inspect_data.py /content/data\nSalin seluruh keluarannya ke chat jika ada yang perlu dicek.\"\"\"\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import is_image, looks_like_mask  # noqa: E402\n\nroot = Path(sys.argv[1] if len(sys.argv) > 1 else \"/content/data\")\nfiles = [p for p in root.rglob(\"*\") if p.is_file() and \"__MACOSX\" not in p.parts and not p.name.startswith(\"._\")]\nprint(f\"Root: {root}\\nTotal file: {len(files)}\\n\")\n\next = Counter(p.suffix.lower() for p in files)\nprint(\"Ekstensi:\", dict(ext.most_common()))\n\nimgs = [p for p in files if is_image(p)]\nmasks = [p for p in imgs if looks_like_mask(p)]\nprint(f\"Gambar: {len(imgs)} | terdeteksi sebagai mask: {len(masks)} | foto asli: {len(imgs) - len(masks)}\\n\")\n\nprint(\"Struktur folder (maks. kedalaman 3, jumlah gambar):\")\ntree = defaultdict(int)\nfor p in imgs:\n    rel = p.relative_to(root).parts[:-1]\n    for d in range(1, min(len(rel), 3) + 1):\n        tree[\"/\".join(rel[:d])] += 1\nchildren = defaultdict(list)\nfor k in tree:\n    children[k.rsplit(\"/\", 1)[0] if \"/\" in k else \"\"].append(k)\n\n\ndef show(parent, depth=0):\n    kids = sorted(children.get(parent, []), key=lambda x: (len(x), x))\n    for j, k in enumerate(kids):\n        if j == 4 and len(kids) > 6:\n            print(\"  \" * depth + f\"- ... (+{len(kids) - 4} folder lain, total {len(kids)})\")\n            break\n        print(\"  \" * depth + f\"- {k.split('/')[-1]}  ({tree[k]} gambar)\")\n        show(k, depth + 1)\n\n\nshow(\"\")\n\nprint(\"\\nContoh nama file (10 pertama):\")\nfor p in imgs[:10]:\n    print(\"  \", p.relative_to(root))\n\nprint(\"\\nContoh ukuran gambar:\")\nfor p in imgs[:5]:\n    try:\n        with Image.open(p) as im:\n            print(f\"   {p.name}: {im.size} {im.mode}\")\n    except Exception as e:  # noqa: BLE001\n        print(f\"   {p.name}: gagal dibuka ({e})\")\n\ntables = [p for p in files if p.suffix.lower() in {\".csv\", \".xlsx\", \".xls\", \".tsv\"}]\nprint(f\"\\nTabel/label ditemukan: {len(tables)}\")\nfor t in tables[:10]:\n    print(f\"\\n== {t.relative_to(root)}\")\n    try:\n        if t.suffix.lower() in {\".xlsx\", \".xls\"}:\n            sheets = pd.read_excel(t, sheet_name=None)\n            for name, df in sheets.items():\n                print(f\"   sheet '{name}': {df.shape}, kolom = {list(df.columns)}\")\n                print(df.head(3).to_string(max_cols=12)[:900])\n        else:\n            df = pd.read_csv(t, sep=None, engine=\"python\")\n            print(f\"   {df.shape}, kolom = {list(df.columns)}\")\n            print(df.head(3).to_string(max_cols=12)[:900])\n    except Exception as e:  # noqa: BLE001\n        print(\"   gagal dibaca:\", e)\n", "build_manifest.py": "\"\"\"Langkah 4: ubah dataset (struktur apa pun) menjadi satu tabel manifest.csv.\n\nYang dikenali otomatis:\n  * Gaya Eyes-defy-anemia: folder per pasien (1, 2, 3 ...) berisi foto + mask\n    *_palpebral.png, dan file Excel/CSV berisi nomor pasien + Hb + jenis kelamin + usia.\n  * Gaya folder kelas: .../anemia/xxx.jpg dan .../normal|non_anemic/xxx.jpg\n  * Gaya tabel: CSV/Excel berisi nama file gambar + Hb atau label.\nDuplikat (gambar sama/hampir sama di beberapa sumber) dideteksi dengan perceptual\nhash dan dibuang, supaya tidak bocor antara data latih dan data uji.\n\nPakai:  python anemialens/build_manifest.py /content/data /content/work\n\"\"\"\nimport re\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import anemia_threshold, is_image, looks_like_mask  # noqa: E402\n\nHB_RE = re.compile(r\"(^|[^a-z])(hb|hgb|hgb_?g?dl|haemoglobin|hemoglobin|hemoglobin.*|haemo.*)([^a-z]|$)\", re.I)\nSEX_RE = re.compile(r\"sex|gender|jenis.?kelamin|^jk$|^m/f$\", re.I)\nAGE_RE = re.compile(r\"(^|[^a-z])(age|usia|umur)([^a-z]|$)\", re.I)\nLABEL_RE = re.compile(r\"anem|remark|class|label|status|diagnos|kategori|category\", re.I)\nFILE_RE = re.compile(r\"file|image|img|name|nama|path|foto|photo|picture\", re.I)\n\n\ndef norm_key(s) -> str:\n    s = str(s).strip().lower()\n    s = re.sub(r\"\\.(jpe?g|png|bmp|tiff?|webp)$\", \"\", s)\n    return s.split(\"/\")[-1].split(\"\\\\\")[-1]\n\n\ndef id_key(s) -> str:\n    \"\"\"'7', '7.0', 'P007', 'patient 7' -> '7'\"\"\"\n    m = re.findall(r\"\\d+\", str(s))\n    return str(int(m[-1])) if m else str(s).strip().lower()\n\n\ndef parse_label(v):\n    if pd.isna(v):\n        return np.nan\n    s = str(v).strip().lower()\n    if s in {\"0\", \"0.0\", \"false\", \"no\", \"n\", \"tidak\"}:\n        return 0\n    if s in {\"1\", \"1.0\", \"true\", \"yes\", \"y\", \"ya\"}:\n        return 1\n    if any(k in s for k in (\"non\", \"not\", \"normal\", \"healthy\", \"sehat\", \"tidak\")):\n        return 0\n    if \"anem\" in s or \"positive\" in s or \"pale\" in s:\n        return 1\n    return np.nan\n\n\nCLASS_NEG = re.compile(r\"^(non[-_ ]?an(a)?emi(a|c)|not[-_ ]?an(a)?emi(a|c)|normal|healthy|sehat|negative|0)$\", re.I)\nCLASS_POS = re.compile(r\"^(an(a)?emi(a|c)|anemia[-_ ]?positive|positive|pale|1)$\", re.I)\n\n\ndef folder_label(parts):\n    \"\"\"Label dari nama folder HANYA jika nama folder persis nama kelas\n    (mis. 'Anemic', 'Non-anemic', 'normal'); 'dataset anemia' tidak dihitung.\"\"\"\n    for part in reversed(parts):\n        s = part.strip()\n        if CLASS_NEG.match(s):\n            return 0\n        if CLASS_POS.match(s):\n            return 1\n    return np.nan\n\n\ndef img_signature(path, size=16):\n    \"\"\"Perceptual hash 16x16 (256 bit) + rata-rata warna + rasio aspek.\n    Dua gambar dianggap duplikat hanya jika KETIGANYA sangat mirip, supaya foto\n    konjungtiva berbeda pasien yang kebetulan mirip tidak ikut terbuang.\"\"\"\n    try:\n        with Image.open(path) as im:\n            rgb = im.convert(\"RGB\")\n            w, h = rgb.size\n            g = rgb.convert(\"L\").resize((size + 1, size), Image.BILINEAR)\n            a = np.asarray(g, dtype=np.int16)\n            px = np.asarray(rgb.resize((64, 64)), dtype=np.float32).reshape(-1, 3)\n            fg = px[px.sum(1) > 15]\n            mean = (fg if len(fg) else px).mean(0)\n        return (a[:, 1:] > a[:, :-1]).flatten(), mean, (w, h)\n    except Exception:  # noqa: BLE001\n        return None\n\n\ndef pick_mask(orig: Path, masks_in_dir):\n    \"\"\"Pilih mask untuk foto: prioritas palpebral murni > forniceal_palpebral > mask lain.\"\"\"\n    if not masks_in_dir:\n        return None\n    stem = orig.stem.lower()\n    same = [m for m in masks_in_dir if m.stem.lower().startswith(stem)] or masks_in_dir\n\n    def rank(m):\n        # normalisasi salah ketik di dataset asli (\"papebral\", \"palplebral\")\n        s = re.sub(r\"pal?p?l?ebral\", \"palpebral\", m.stem.lower())\n        if \"palpebral\" in s and \"forniceal\" not in s:\n            return 0\n        if \"forniceal_palpebral\" in s or (\"palpebral\" in s and \"forniceal\" in s):\n            return 1\n        if \"forniceal\" in s:\n            return 3\n        return 2\n    return sorted(same, key=rank)[0]\n\n\ndef load_tables(root: Path):\n    out = []\n    for t in root.rglob(\"*\"):\n        if t.suffix.lower() not in {\".csv\", \".xlsx\", \".xls\", \".tsv\"} or \"__MACOSX\" in t.parts:\n            continue\n        try:\n            if t.suffix.lower() in {\".xlsx\", \".xls\"}:\n                for name, df in pd.read_excel(t, sheet_name=None).items():\n                    out.append((t, name, df))\n            else:\n                out.append((t, \"\", pd.read_csv(t, sep=None, engine=\"python\")))\n        except Exception as e:  # noqa: BLE001\n            print(f\"  [lewati] {t.name}: {e}\")\n    return out\n\n\ndef find_col(df, regex, exclude=()):\n    for c in df.columns:\n        if c in exclude:\n            continue\n        if regex.search(str(c)):\n            return c\n    return None\n\n\ndef main(data_root, work_dir):\n    root, work = Path(data_root), Path(work_dir)\n    work.mkdir(parents=True, exist_ok=True)\n    all_imgs = [p for p in root.rglob(\"*\") if p.is_file() and is_image(p)\n                and \"__MACOSX\" not in p.parts and not p.name.startswith(\"._\")]\n    masks = [p for p in all_imgs if looks_like_mask(p)]\n    origs = [p for p in all_imgs if not looks_like_mask(p)]\n    by_dir = defaultdict(list)\n    for m in masks:\n        by_dir[m.parent].append(m)\n    origs_per_dir = defaultdict(int)\n    for o in origs:\n        origs_per_dir[o.parent] += 1\n    print(f\"Foto asli: {len(origs)} | mask: {len(masks)}\")\n\n    rel_dirs = [o.relative_to(root).parts[:-1] for o in origs]\n    common = 0\n    if rel_dirs:\n        while all(len(r) > common for r in rel_dirs) and len({r[common] for r in rel_dirs}) == 1:\n            common += 1\n\n    rows = []\n    for o in origs:\n        parts = o.relative_to(root).parts[:-1]\n        source = parts[common] if len(parts) > common else (parts[-1] if parts else \"root\")\n        # folder per pasien: nama folder berupa angka, atau folder kecil berisi <=6 foto\n        dname = o.parent.name\n        per_patient_dir = bool(re.fullmatch(r\"\\d+\", dname)) or (\n            origs_per_dir[o.parent] <= 6 and o.parent in by_dir)\n        if per_patient_dir:\n            # sumber = dataset + negara/subset (folder induk pasien), mis. \"eyes_defy_anemia/India\"\n            if len(parts) >= 2 and len(parts) - 1 > common:\n                source = f\"{source}/{parts[-2]}\" if parts[-2] != source else source\n            pid = \"pasien::\" + \"/\".join(parts)          # path folder pasien = ID unik\n        else:\n            pid = f\"{source}::{o.stem}\"\n        m = pick_mask(o, by_dir.get(o.parent, []))\n        rows.append(dict(image_path=str(o), mask_path=str(m) if m else \"\", source=source,\n                         patient_id=pid, patient_dir=str(o.parent) if per_patient_dir else \"\",\n                         folder_label=folder_label(parts), hb=np.nan, sex=np.nan, age=np.nan,\n                         table_label=np.nan))\n    df = pd.DataFrame(rows)\n    df[\"sex\"] = df[\"sex\"].astype(object)\n    df[\"mask_kind\"] = np.where(df.mask_path != \"\", \"manual\", \"\")\n    # Foto yang SUDAH tersegmentasi dengan latar transparan (mis. CP-AnemiC):\n    # kanal alfa dijadikan mask. Mask ini tidak dipakai untuk melatih U-Net.\n    amask_dir = work / \"alpha_masks\"\n    n_alpha = 0\n    for i in df.index[df.mask_path == \"\"]:\n        try:\n            with Image.open(df.at[i, \"image_path\"]) as im:\n                if im.mode not in (\"RGBA\", \"LA\", \"PA\") and \"transparency\" not in im.info:\n                    continue\n                a = np.asarray(im.convert(\"RGBA\"))[:, :, 3]\n        except Exception:  # noqa: BLE001\n            continue\n        if 0.05 < (a == 0).mean() < 0.98:\n            amask_dir.mkdir(exist_ok=True)\n            mp = amask_dir / f\"{i:06d}.png\"\n            Image.fromarray(((a > 0) * 255).astype(np.uint8)).save(mp)\n            df.at[i, \"mask_path\"], df.at[i, \"mask_kind\"] = str(mp), \"alpha\"\n            n_alpha += 1\n    if n_alpha:\n        print(f\"  {n_alpha} foto sudah tersegmentasi (latar transparan) -> mask dari kanal alfa\")\n    if df.empty:\n        sys.exit(\"Tidak ada gambar ditemukan. Cek path dataset.\")\n\n    # ------------------------------------------------------------ gabung tabel\n    stem_idx = defaultdict(list)\n    for i, p in enumerate(df.image_path):\n        stem_idx[norm_key(p)].append(i)\n    for t, sheet, tab in load_tables(root):\n        tab = tab.dropna(how=\"all\")\n        if tab.empty:\n            continue\n        hb_c = find_col(tab, HB_RE)\n        sex_c = find_col(tab, SEX_RE)\n        age_c = find_col(tab, AGE_RE, exclude=(sex_c,))\n        lab_c = find_col(tab, LABEL_RE, exclude=(hb_c, sex_c, age_c))\n        if hb_c is None and lab_c is None:\n            continue\n        best = (0, None, None)\n        # a) gabung lewat nama file\n        for c in tab.columns:\n            keys = tab[c].astype(str).map(norm_key)\n            hits = sum(k in stem_idx for k in keys)\n            if hits > best[0]:\n                best = (hits, c, \"file\")\n        # b) gabung lewat nomor pasien = nama folder pasien di bawah lokasi tabel\n        under = df[df.image_path.str.startswith(str(t.parent)) & (df.patient_dir != \"\")]\n        dir_ids = defaultdict(list)\n        for i, d in zip(under.index, under.patient_dir):\n            dir_ids[id_key(Path(d).name)].append(i)\n        if dir_ids:\n            for c in tab.columns:\n                if c in (hb_c, sex_c, age_c):\n                    continue\n                keys = tab[c].map(id_key)\n                hits = sum(k in dir_ids for k in keys)\n                if hits > best[0]:\n                    best = (hits, c, \"id\")\n        hits, key_c, mode = best\n        if hits < max(3, 0.3 * min(len(tab), 50)):\n            print(f\"  [tabel tidak tertaut] {t.name}{'/' + sheet if sheet else ''}: kolom Hb={hb_c}, label={lab_c}\")\n            continue\n        print(f\"  [tabel] {t.name}{'/' + sheet if sheet else ''}: tertaut lewat '{key_c}' ({mode}, {hits} baris) \"\n              f\"| Hb='{hb_c}' sex='{sex_c}' age='{age_c}' label='{lab_c}'\")\n        for _, r in tab.iterrows():\n            idxs = stem_idx.get(norm_key(r[key_c]), []) if mode == \"file\" else dir_ids.get(id_key(r[key_c]), [])\n            for i in idxs:\n                if hb_c is not None:\n                    v = pd.to_numeric(str(r[hb_c]).replace(\",\", \".\"), errors=\"coerce\")\n                    if pd.notna(v) and 2 < v < 25:\n                        df.at[i, \"hb\"] = float(v)\n                if sex_c is not None and pd.notna(r[sex_c]):\n                    df.at[i, \"sex\"] = str(r[sex_c]).strip()\n                if age_c is not None:\n                    df.at[i, \"age\"] = pd.to_numeric(r[age_c], errors=\"coerce\")\n                if lab_c is not None:\n                    df.at[i, \"table_label\"] = parse_label(r[lab_c])\n\n    # ------------------------------------------------------------ label akhir\n    # Dataset balita (mis. CP-AnemiC, 6-59 bulan) mencatat usia dalam BULAN.\n    # Deteksi: semua usia di satu sumber <= 60 dan ada yang > 18 -> anggap bulan.\n    df[\"age_years\"] = df.age\n    for src, g in df.groupby(\"source\"):\n        a = g.age.dropna()\n        if len(a) and a.max() <= 60 and a.min() < 12 and (a > 18).any():\n            df.loc[g.index, \"age_years\"] = g.age / 12\n            print(f\"  [usia] sumber '{src}' dianggap dalam bulan (maks {a.max():.0f}).\")\n\n    def final_label(r):\n        if pd.notna(r.hb):\n            return int(r.hb < anemia_threshold(r.age_years, r.sex)), \"hb_lab\"\n        if pd.notna(r.table_label):\n            return int(r.table_label), \"tabel\"\n        if pd.notna(r.folder_label):\n            return int(r.folder_label), \"folder\"\n        return -1, \"tidak_ada\"\n    lab = df.apply(final_label, axis=1, result_type=\"expand\")\n    df[\"label\"], df[\"label_source\"] = lab[0], lab[1]\n\n    # ------------------------------------------------------------ duplikat\n    print(\"Mengecek duplikat (perceptual hash)...\")\n    sigs = [img_signature(p) for p in df.image_path]\n    ok = [i for i, h in enumerate(sigs) if h is not None]\n    H = np.array([sigs[i][0] for i in ok], dtype=bool)\n    MC = np.array([sigs[i][1] for i in ok], dtype=np.float32)\n    WH = np.array([sigs[i][2] for i in ok], dtype=np.float32)\n    drop, group_of = set(), {}\n    for a in range(len(ok)):\n        ia = ok[a]\n        if ia in drop:\n            continue\n        d = (H[a + 1:] != H[a]).sum(axis=1)\n        dc = np.abs(MC[a + 1:] - MC[a]).max(axis=1)\n        dwh = np.abs(WH[a + 1:] - WH[a]).max(axis=1)\n        # duplikat = ukuran sama, struktur hampir identik, warna latar-depan hampir identik\n        for b in np.where((d <= 3) & (dc < 1.5) & (dwh <= 2))[0]:\n            ib = ok[a + 1 + b]\n            if ib not in drop:\n                drop.add(ib)\n                group_of[ib] = ia\n    # Kelompokkan duplikat. Jika satu gambar yang SAMA dipakai untuk beberapa pasien\n    # dengan Hb/label berbeda, pasangan gambar-Hb itu tidak bisa dipercaya -> buang semua.\n    groups = defaultdict(list)\n    for ib, ia in group_of.items():\n        groups[ia].append(ib)\n    df[\"duplicate_of\"], df[\"drop_reason\"] = \"\", \"\"\n    n_incons = 0\n    for ia, members in groups.items():\n        allm = [ia] + members\n        hbv = df.loc[allm, \"hb\"].dropna()\n        incons = (df.loc[allm, \"label\"].nunique() > 1) or (len(hbv) > 1 and hbv.max() - hbv.min() > 0.5)\n        for ib in members:\n            df.at[ib, \"duplicate_of\"] = df.at[ia, \"image_path\"]\n            df.at[ib, \"drop_reason\"] = \"duplikat\"\n        if incons:\n            n_incons += 1\n            for i in allm:\n                df.at[i, \"drop_reason\"] = \"gambar_sama_hb_berbeda\"\n    dups = df[df.drop_reason != \"\"]\n    df = df[df.drop_reason == \"\"].reset_index(drop=True)\n    print(f\"  grup gambar identik: {len(groups)} | dibuang total: {len(dups)} foto\")\n    if n_incons:\n        print(f\"  PERHATIAN: {n_incons} grup berisi gambar yang sama untuk pasien dengan Hb berbeda. \"\n              f\"Seluruh anggota grup dibuang ({(dups.drop_reason == 'gambar_sama_hb_berbeda').sum()} foto).\")\n        print(\"  Per sumber:\", dups.groupby('source').size().to_dict())\n    dups.to_csv(work / \"duplicates.csv\", index=False)\n\n    unlabeled = (df.label < 0).sum()\n    df = df[df.label >= 0].reset_index(drop=True)\n    df.to_csv(work / \"manifest.csv\", index=False)\n\n    print(\"\\n================ RINGKASAN MANIFEST ================\")\n    print(f\"Total foto berlabel: {len(df)} (tanpa label dibuang: {unlabeled})\")\n    print(f\"Pasien unik: {df.patient_id.nunique()} | punya mask: {(df.mask_path != '').sum()} | punya Hb: {df.hb.notna().sum()}\")\n    summ = df.groupby(\"source\").agg(foto=(\"image_path\", \"size\"), pasien=(\"patient_id\", \"nunique\"),\n                                    anemia=(\"label\", \"sum\"), mask=(\"mask_path\", lambda s: (s != \"\").sum()),\n                                    hb=(\"hb\", lambda s: s.notna().sum()), hb_mean=(\"hb\", \"mean\"))\n    print(summ.to_string())\n    print(\"\\nSumber label:\", df.label_source.value_counts().to_dict())\n    print(f\"\\nTersimpan: {work / 'manifest.csv'}\")\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 else \"/content/data\",\n         sys.argv[2] if len(sys.argv) > 2 else \"/content/work\")\n", "train_seg.py": "\"\"\"Langkah 5: latih model segmentasi konjungtiva (U-Net) dari mask yang tersedia,\nlalu buat mask untuk foto yang belum punya mask.\n\nPakai:  python anemialens/train_seg.py /content/work\n        python anemialens/train_seg.py /content/work --roi-sources=CP-AnemiC   (paksa sumber = sudah ter-crop)\nHasil:  work/seg_unet.pt, work/pred_masks/*.png, kolom roi_mask & roi_mode di manifest.csv\n\"\"\"\nimport sys\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import Config, read_mask, read_rgb, seed_everything  # noqa: E402\n\nMEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nSTD = np.array([0.229, 0.224, 0.225], dtype=np.float32)\n\n\ndef build_unet(pretrained=True):\n    import segmentation_models_pytorch as smp\n    return smp.Unet(encoder_name=\"resnet34\", encoder_weights=\"imagenet\" if pretrained else None,\n                    in_channels=3, classes=1)\n\n\ndef to_tensor(img):\n    x = (img.astype(np.float32) / 255.0 - MEAN) / STD\n    return torch.from_numpy(x.transpose(2, 0, 1))\n\n\ndef augment(img, m, rng):\n    if rng.random() < 0.5:\n        img, m = img[:, ::-1], m[:, ::-1]\n    h, w = m.shape\n    ang = rng.uniform(-20, 20)\n    sc = rng.uniform(0.85, 1.15)\n    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)\n    M[:, 2] += rng.uniform(-0.06, 0.06, 2) * [w, h]\n    img = cv2.warpAffine(np.ascontiguousarray(img), M, (w, h), borderMode=cv2.BORDER_REFLECT)\n    m = cv2.warpAffine(np.ascontiguousarray(m), M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)\n    img = img.astype(np.float32) * rng.uniform(0.75, 1.25) + rng.uniform(-20, 20)   # kecerahan/kontras\n    img = img * rng.uniform(0.92, 1.08, 3)                                           # beda white balance kamera\n    return np.clip(img, 0, 255).astype(np.uint8), m\n\n\ndef _feather(m, k):\n    k = max(3, int(k) | 1)\n    return cv2.GaussianBlur(m.astype(np.float32), (k, k), 0)[..., None]\n\n\ndef pallor(img, m, rng, strength=None):\n    \"\"\"Simulasi konjungtiva PUCAT (anemia): geser warna area mask ke arah merah-muda keputihan\n    (CIELAB: lebih terang, kemerahan a* turun tetapi tetap positif), tekstur pembuluh tetap ada,\n    tepi halus. Mengajari U-Net bahwa konjungtiva pucat tetap konjungtiva.\"\"\"\n    if not m.any():\n        return img\n    t = rng.uniform(0.35, 0.85) if strength is None else strength\n    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)\n    sel = m.astype(bool)\n    mean = lab[sel].mean(0)\n    pale = np.array([rng.uniform(175, 215), rng.uniform(136, 146), rng.uniform(122, 132)], np.float32)\n    tgt = lab + t * (pale - mean)                       # geser rata-rata, tekstur dipertahankan\n    tgt[..., 1:] = 128 + (tgt[..., 1:] - 128) * (1 - 0.3 * t) + 0.3 * t * (pale[1:] - 128)\n    w = _feather(m, max(m.shape) * 0.02)\n    out = lab * (1 - w) + tgt * w\n    return cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)\n\n\ndef _soft_background(img):\n    \"\"\"Latar untuk zoom-out: versi sangat buram dari foto itu sendiri (warna kulit/sekitar, tanpa bentuk).\"\"\"\n    h, w = img.shape[:2]\n    small = cv2.resize(img, (max(4, w // 16), max(4, h // 16)), interpolation=cv2.INTER_AREA)\n    return cv2.GaussianBlur(cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR), (0, 0), max(h, w) / 20)\n\n\ndef augment_robust(img, m, rng):\n    \"\"\"Augmentasi 'foto HP dunia nyata': zoom-out (mata kecil di bingkai), rotasi/perspektif,\n    konjungtiva pucat, white balance & saturasi berbeda, blur, noise, kompresi JPEG, resolusi rendah.\"\"\"\n    h, w = m.shape\n    if rng.random() < 0.5:\n        img, m = img[:, ::-1], m[:, ::-1]\n    img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)\n    if rng.random() < 0.5:\n        img = pallor(img, m, rng)\n    # geometri: skala 0.35-1.3 (zoom-out kuat), rotasi, geser, sedikit perspektif\n    sc = rng.uniform(0.35, 1.0) if rng.random() < 0.55 else rng.uniform(1.0, 1.3)\n    ang = rng.uniform(-30, 30)\n    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)\n    M[:, 2] += rng.uniform(-0.2, 0.2, 2) * [w, h]\n    M3 = np.vstack([M, [0, 0, 1]])\n    if rng.random() < 0.4:\n        d = 0.06 * w\n        src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])\n        dst = src + rng.uniform(-d, d, (4, 2)).astype(np.float32)\n        M3 = cv2.getPerspectiveTransform(src, dst) @ M3\n    bg = _soft_background(img)\n    valid = cv2.warpPerspective(np.ones((h, w), np.float32), M3, (w, h), borderMode=cv2.BORDER_CONSTANT)\n    valid = cv2.GaussianBlur(valid, (0, 0), 2)[..., None]\n    img = cv2.warpPerspective(img, M3, (w, h), borderMode=cv2.BORDER_CONSTANT).astype(np.float32)\n    img = (img * valid + bg.astype(np.float32) * (1 - valid)).astype(np.uint8)\n    m = cv2.warpPerspective(m, M3, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)\n    # warna & cahaya\n    x = img.astype(np.float32)\n    x = x * rng.uniform(0.85, 1.15, 3)                                   # white balance\n    x = x * rng.uniform(0.7, 1.3) + rng.uniform(-25, 25)                  # kontras/kecerahan\n    x = 255 * (np.clip(x, 0, 255) / 255) ** rng.uniform(0.7, 1.4)         # gamma\n    hsv = cv2.cvtColor(np.clip(x, 0, 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)\n    hsv[..., 1] *= rng.uniform(0.6, 1.3)\n    hsv[..., 0] = (hsv[..., 0] + rng.uniform(-4, 4)) % 180\n    img = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB)\n    # kualitas kamera\n    if rng.random() < 0.3:\n        k = int(rng.choice([3, 5, 7]))\n        img = cv2.GaussianBlur(img, (k, k), 0)\n    if rng.random() < 0.3:\n        f = rng.uniform(0.25, 0.6)\n        small = cv2.resize(img, (max(16, int(w * f)), max(16, int(h * f))), interpolation=cv2.INTER_AREA)\n        img = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)\n    if rng.random() < 0.3:\n        img = np.clip(img + rng.normal(0, rng.uniform(3, 10), img.shape), 0, 255).astype(np.uint8)\n    if rng.random() < 0.5:\n        ok, enc = cv2.imencode(\".jpg\", img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.uniform(30, 90))])\n        img = cv2.imdecode(enc, cv2.IMREAD_UNCHANGED)\n    return img, m\n\n\nclass SegDS(Dataset):\n    def __init__(self, df, size, train, robust=False, fixed_seed=None):\n        self.df, self.size, self.train, self.robust = df.reset_index(drop=True), size, train, robust\n        self.fixed_seed = fixed_seed   # untuk validasi 'foto HP' yang sama setiap epoch\n        self.rng = np.random.default_rng()\n\n    def __len__(self):\n        return len(self.df)\n\n    _cache = {}   # foto 12 MP cukup di-decode & diperkecil sekali (per worker)\n\n    def __getitem__(self, i):\n        r = self.df.iloc[i]\n        key = (r.image_path, self.size)\n        if key not in self._cache:\n            img = read_rgb(r.image_path)\n            m = read_mask(r.mask_path, img.shape)\n            self._cache[key] = (cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_AREA),\n                                cv2.resize(m, (self.size, self.size), interpolation=cv2.INTER_NEAREST))\n        img, m = (a.copy() for a in self._cache[key])\n        if self.fixed_seed is not None:\n            img, m = augment_robust(img, m, np.random.default_rng(self.fixed_seed + i))\n        elif self.train:\n            img, m = (augment_robust if self.robust and self.rng.random() < 0.85 else augment)(img, m, self.rng)\n        return to_tensor(img), torch.from_numpy(m.astype(np.float32))[None]\n\n\ndef dice_loss(logits, target, eps=1.0):\n    p = torch.sigmoid(logits)\n    inter = (p * target).sum((1, 2, 3))\n    return 1 - ((2 * inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps)).mean()\n\n\ndef dice_score(logits, target):\n    p = (torch.sigmoid(logits) > 0.5).float()\n    inter = (p * target).sum((1, 2, 3))\n    return ((2 * inter + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)).mean().item()\n\n\ndef largest_component(m):\n    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)\n    if n <= 1:\n        return m\n    k = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])\n    return (lab == k).astype(np.uint8)\n\n\n@torch.no_grad()\ndef predict_mask(model, img, size, device):\n    \"\"\"Prediksi mask + TTA flip horizontal, dikembalikan ke ukuran asli.\"\"\"\n    x = to_tensor(cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA))[None].to(device)\n    p = torch.sigmoid(model(x)) + torch.sigmoid(model(torch.flip(x, [3]))).flip(3)\n    p = (p / 2)[0, 0].float().cpu().numpy()\n    m = (cv2.resize(p, (img.shape[1], img.shape[0])) > 0.5).astype(np.uint8)\n    return largest_component(m)\n\n\ndef flag(name, default=None):\n    for a in sys.argv:\n        if a == f\"--{name}\":\n            return True\n        if a.startswith(f\"--{name}=\"):\n            return a.split(\"=\", 1)[1]\n    return default\n\n\ndef main(work_dir):\n    work = Path(work_dir)\n    cfg = Config.load(work / \"config.json\") if (work / \"config.json\").exists() else Config(work_dir=str(work))\n    robust = bool(flag(\"robust\", False))\n    epochs = int(flag(\"epochs\", cfg.seg_epochs))\n    seed_everything(cfg.seed)\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    df = pd.read_csv(work / \"manifest.csv\", keep_default_na=False)\n    df[\"mask_path\"] = df.mask_path.astype(str)\n    kind = df[\"mask_kind\"] if \"mask_kind\" in df else pd.Series(np.where(df.mask_path != \"\", \"manual\", \"\"))\n    has = df[(df.mask_path != \"\") & (kind == \"manual\")]   # hanya mask foto mata utuh\n    print(f\"Foto dengan mask: {len(has)} dari {len(df)} | device: {device}\")\n    ckpt = work / \"seg_unet.pt\"\n\n    if len(has) >= 20:\n        pats = has.patient_id.unique()\n        rng = np.random.default_rng(cfg.seed)\n        val_p = set(rng.choice(pats, size=max(2, int(0.15 * len(pats))), replace=False))\n        tr, va = has[~has.patient_id.isin(val_p)], has[has.patient_id.isin(val_p)]\n        dl_tr = DataLoader(SegDS(tr, cfg.seg_size, True, robust=robust), batch_size=8, shuffle=True,\n                           num_workers=cfg.num_workers, drop_last=len(tr) > 8)\n        dl_va = DataLoader(SegDS(va, cfg.seg_size, False), batch_size=8, num_workers=cfg.num_workers)\n        # validasi kedua: foto validasi yang sama tetapi 'dibuat seperti foto HP' (zoom-out, pucat, blur, JPEG)\n        dl_hp = DataLoader(SegDS(va, cfg.seg_size, False, fixed_seed=123), batch_size=8, num_workers=cfg.num_workers)\n        print(f\"Mode augmentasi: {'ROBUST (foto HP + konjungtiva pucat)' if robust else 'standar'} | epoch: {epochs}\")\n        model = build_unet(cfg.pretrained).to(device)\n        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=epochs * len(dl_tr))\n        scaler = torch.amp.GradScaler(enabled=device == \"cuda\")\n        best, best_pair = -1, (0.0, 0.0)\n        for ep in range(epochs):\n            model.train()\n            t0, tl = time.time(), 0.0\n            for x, y in dl_tr:\n                x, y = x.to(device), y.to(device)\n                with torch.autocast(device_type=device, enabled=device == \"cuda\"):\n                    out = model(x)\n                    loss = F.binary_cross_entropy_with_logits(out, y) + dice_loss(out, y)\n                opt.zero_grad()\n                scaler.scale(loss).backward()\n                scaler.step(opt)\n                scaler.update()\n                sched.step()\n                tl += loss.item()\n            model.eval()\n\n            def evaluate(dl):\n                with torch.no_grad():\n                    return float(np.mean([dice_score(model(x.to(device)), y.to(device)) for x, y in dl]))\n            d, dh = evaluate(dl_va), evaluate(dl_hp)\n            score = (d + dh) / 2 if robust else d       # mode robust: pilih model yang bagus di keduanya\n            if score > best:\n                best, best_pair = score, (d, dh)\n                torch.save(model.state_dict(), ckpt)\n            print(f\"  epoch {ep + 1:02d}/{epochs} loss {tl / len(dl_tr):.3f} | Dice val {d:.3f} | Dice 'foto HP' {dh:.3f} \"\n                  f\"| terbaik {best:.3f} | {time.time() - t0:.0f}s\")\n        print(f\"Segmentasi selesai. Dice validasi: {best_pair[0]:.3f} | Dice simulasi foto HP: {best_pair[1]:.3f}\")\n        (work / \"seg_metrics.txt\").write_text(f\"dice_val={best_pair[0]:.4f}\\ndice_val_hp={best_pair[1]:.4f}\\n\"\n                                              f\"robust={int(robust)}\\nn_train={len(tr)}\\nn_val={len(va)}\\n\")\n    elif ckpt.exists():\n        print(\"Mask kurang dari 20, memakai checkpoint segmentasi yang sudah ada.\")\n    else:\n        print(\"PERINGATAN: tidak cukup mask untuk melatih segmentasi. Semua foto dipakai utuh (roi_mode=full).\")\n\n    if flag(\"no-predict\", False):\n        return\n    # ------------------------------------------------ buat mask untuk semua foto\n    out_dir = work / \"pred_masks\"\n    out_dir.mkdir(exist_ok=True)\n    roi_mask, roi_mode, pred_area = [], [], []\n    model = None\n    if ckpt.exists():\n        model = build_unet(False).to(device)\n        model.load_state_dict(torch.load(ckpt, map_location=device))\n        model.eval()\n    for i, r in df.iterrows():\n        if r.mask_path:\n            roi_mask.append(r.mask_path)\n            roi_mode.append(\"gt\" if getattr(r, \"mask_kind\", \"manual\") != \"alpha\" else \"alpha\")\n            pred_area.append(np.nan)\n            continue\n        if model is None:\n            roi_mask.append(\"\")\n            roi_mode.append(\"full\")\n            pred_area.append(np.nan)\n            continue\n        img = read_rgb(r.image_path)\n        m = predict_mask(model, img, cfg.seg_size, device)\n        area = float(m.mean())\n        pred_area.append(area)\n        # Foto yang sudah berupa potongan konjungtiva (mask menutup hampir seluruh gambar)\n        # atau segmentasi gagal -> pakai gambar utuh.\n        if area >= 0.45 or area < 0.005:\n            roi_mask.append(\"\")\n            roi_mode.append(\"full\")\n        else:\n            p = out_dir / f\"{i:06d}.png\"\n            cv2.imwrite(str(p), m * 255)\n            roi_mask.append(str(p))\n            roi_mode.append(\"pred\")\n    df[\"roi_mask\"], df[\"roi_mode\"], df[\"pred_mask_area\"] = roi_mask, roi_mode, pred_area\n    # Sumber tanpa mask asli yang fotonya sudah berupa potongan konjungtiva (mis. CP-AnemiC):\n    # jika median area prediksi besar, perlakukan seluruh sumber sebagai ROI utuh.\n    forced = [a.split(\"=\", 1)[1] for a in sys.argv if a.startswith(\"--roi-sources=\")]\n    forced = set(forced[0].split(\",\")) if forced else set()\n    for src, g in df.groupby(\"source\"):\n        if g.roi_mode.isin([\"gt\", \"alpha\"]).any():\n            continue\n        med = g.pred_mask_area.median()\n        if src in forced or (pd.notna(med) and med >= 0.30):\n            df.loc[g.index, \"roi_mask\"] = \"\"\n            df.loc[g.index, \"roi_mode\"] = \"full\"\n            print(f\"  Sumber '{src}' diperlakukan sebagai foto yang sudah ter-crop (median area {med:.2f}).\")\n    df.to_csv(work / \"manifest.csv\", index=False)\n    print(\"Mode ROI:\", df.roi_mode.value_counts().to_dict())\n    print(\"Per sumber:\\n\", pd.crosstab(df.source, df.roi_mode).to_string())\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith(\"--\") else \"/content/work\")\n", "prepare_rois.py": "\"\"\"Langkah 6: potong area konjungtiva setiap foto, hitung fitur warna & kualitas.\n\nPakai:  python anemialens/prepare_rois.py /content/work\nHasil:  work/rois/*.png (crop konjungtiva, latar hitam), work/features.csv\n\"\"\"\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import (COLOR_FEATURE_NAMES, color_features, crop_roi, letterbox,  # noqa: E402\n                               quality_check, read_mask, read_rgb)\n\nSTORE = 448  # ukuran simpan ROI (classifier memakai 224)\n\n\ndef main(work_dir):\n    work = Path(work_dir)\n    df = pd.read_csv(work / \"manifest.csv\", keep_default_na=False)\n    out = work / \"rois\"\n    out.mkdir(exist_ok=True)\n    from anemialens.roi_guard import mask_shape\n    feats, qual, paths, shapes = [], [], [], []\n    for i, r in df.iterrows():\n        img = read_rgb(r.image_path)\n        m = read_mask(r.roi_mask, img.shape) if r.roi_mask else None\n        q = quality_check(img, m)\n        roi = crop_roi(img, m)\n        feats.append(color_features(roi))\n        shapes.append(mask_shape(m) if m is not None else (np.nan, np.nan))\n        qual.append(q)\n        p = out / f\"{i:06d}.png\"\n        cv2.imwrite(str(p), cv2.cvtColor(letterbox(roi, STORE), cv2.COLOR_RGB2BGR))\n        paths.append(str(p))\n        if (i + 1) % 200 == 0:\n            print(f\"  {i + 1}/{len(df)}\")\n    F = pd.DataFrame(np.vstack(feats), columns=COLOR_FEATURE_NAMES)\n    Q = pd.DataFrame(qual)\n    Q[\"mask_area_frac\"], Q[\"mask_elong\"] = [s_[0] for s_ in shapes], [s_[1] for s_ in shapes]\n    Q[\"issues\"] = Q.issues.map(lambda x: \";\".join(x))\n    res = pd.concat([df.reset_index(drop=True), pd.Series(paths, name=\"roi_path\"), F, Q], axis=1)\n    res.to_csv(work / \"features.csv\", index=False)\n    # Ambang kualitas dikalibrasi dari data latih: foto di aplikasi ditolak hanya jika\n    # lebih buruk dari 98% foto dataset (bukan angka tebakan).\n    import json\n    ref = {\"blur_min\": float(np.percentile(Q.blur, 2)), \"bright_min\": float(np.percentile(Q.brightness, 1)),\n           \"bright_max\": float(np.percentile(Q.brightness, 99)),\n           \"glare_max\": float(max(0.08, np.nanpercentile(Q.glare, 99)))}\n    (work / \"quality_ref.json\").write_text(json.dumps(ref, indent=2))\n    print(\"Ambang kualitas terkalibrasi:\", {k: round(v, 3) for k, v in ref.items()})\n    print(f\"Selesai: {len(res)} ROI. Foto dengan masalah kualitas: {(Q.issues != '').sum()}\")\n    print(Q.issues[Q.issues != \"\"].value_counts().head(10).to_string())\n    # Pemeriksaan cepat: apakah fitur warna memang berbeda antara anemia vs normal?\n    print(\"\\nRata-rata fitur kunci per kelas (anemia seharusnya a* & erythema index lebih rendah):\")\n    print(res.groupby(\"label\")[[\"a_mean\", \"erythema_index\", \"high_hue_ratio\", \"L_mean\"]].mean().round(3).to_string())\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 else \"/content/work\")\n", "models.py": "\"\"\"Model AnemiaLens:\n  A. Fitur warna klasik (a*, erythema index, HHR, dst.) -> regresi logistik / ridge\n  A2. Fitur warna yang sama -> Random Forest (menangkap hubungan non-linear)\n  B. Embedding DINOv2 ViT-S/14 (beku) -> regresi logistik / ridge  (linear probe)\n  C. CNN multi-task (EfficientNet-B0) di-fine-tune: kepala klasifikasi anemia + kepala regresi Hb\nPrediksi akhir = rata-rata (ensemble) dari model yang tersedia.\"\"\"\nfrom __future__ import annotations\n\nimport math\nimport time\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nMEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nSTD = np.array([0.229, 0.224, 0.225], dtype=np.float32)\nHB_MU, HB_SD = 12.0, 2.0\n\n\ndef to_tensor(img):\n    return torch.from_numpy(((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1))\n\n\ndef load_roi(path, size):\n    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)\n    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)\n\n\ndef augment_roi(img, rng):\n    \"\"\"Augmentasi yang TIDAK merusak sinyal warna: geometri bebas, tapi perubahan\n    warna dibatasi (tanpa hue jitter, gain per kanal kecil untuk meniru beda kamera).\"\"\"\n    if rng.random() < 0.5:\n        img = img[:, ::-1]\n    if rng.random() < 0.3:\n        img = img[::-1]\n    h, w = img.shape[:2]\n    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-15, 15), rng.uniform(0.9, 1.15))\n    M[:, 2] += rng.uniform(-0.05, 0.05, 2) * [w, h]\n    img = cv2.warpAffine(np.ascontiguousarray(img), M, (w, h), borderMode=cv2.BORDER_CONSTANT)\n    f = img.astype(np.float32)\n    nz = f.sum(axis=2, keepdims=True) > 0\n    f = f * rng.uniform(0.85, 1.15) + rng.uniform(-12, 12)      # kecerahan & kontras\n    f = f * rng.uniform(0.95, 1.05, 3)                           # white balance kamera\n    f = np.where(nz, f, 0)\n    if rng.random() < 0.2:                                       # random erasing kecil\n        eh, ew = int(h * rng.uniform(0.1, 0.25)), int(w * rng.uniform(0.1, 0.25))\n        y, x = rng.integers(0, h - eh), rng.integers(0, w - ew)\n        f[y:y + eh, x:x + ew] = 0\n    return np.clip(f, 0, 255).astype(np.uint8)\n\n\nclass RoiDS(Dataset):\n    def __init__(self, paths, labels, hbs, size, train):\n        self.p, self.y, self.hb, self.size, self.train = list(paths), np.asarray(labels, np.float32), \\\n            np.asarray(hbs, np.float32), size, train\n        self.rng = np.random.default_rng()\n\n    def __len__(self):\n        return len(self.p)\n\n    def __getitem__(self, i):\n        img = load_roi(self.p[i], self.size)\n        if self.train:\n            img = augment_roi(img, self.rng)\n        hb = self.hb[i]\n        has = 0.0 if np.isnan(hb) else 1.0\n        hbn = 0.0 if np.isnan(hb) else (hb - HB_MU) / HB_SD\n        return to_tensor(img), torch.tensor(self.y[i]), torch.tensor(hbn, dtype=torch.float32), torch.tensor(has)\n\n\nclass MultiTaskNet(nn.Module):\n    def __init__(self, backbone=\"efficientnet_b0\", pretrained=True, drop=0.3):\n        super().__init__()\n        import timm\n        self.body = timm.create_model(backbone, pretrained=pretrained, num_classes=0)\n        d = self.body.num_features\n        self.drop = nn.Dropout(drop)\n        self.cls = nn.Linear(d, 1)\n        self.hb = nn.Linear(d, 1)\n\n    def forward(self, x):\n        f = self.drop(self.body(x))\n        return self.cls(f).squeeze(1), self.hb(f).squeeze(1)\n\n\ndef train_cnn(cfg, tr_paths, tr_y, tr_hb, device, log_prefix=\"\"):\n    seed = cfg.seed\n    torch.manual_seed(seed)\n    model = MultiTaskNet(cfg.cnn_backbone, cfg.pretrained).to(device)\n    ds = RoiDS(tr_paths, tr_y, tr_hb, cfg.img_size, True)\n    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,\n                    drop_last=len(ds) > cfg.batch_size, pin_memory=device == \"cuda\")\n    pos = float(np.mean(tr_y))\n    pos_weight = torch.tensor((1 - pos) / max(pos, 1e-3), device=device)\n    body_p = list(model.body.parameters())\n    head_p = list(model.cls.parameters()) + list(model.hb.parameters())\n    opt = torch.optim.AdamW([{\"params\": body_p, \"lr\": cfg.lr * 0.5}, {\"params\": head_p, \"lr\": cfg.lr * 3}],\n                            weight_decay=cfg.weight_decay)\n    total = cfg.cnn_epochs * len(dl)\n    warm = max(1, len(dl))\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))\n    scaler = torch.amp.GradScaler(enabled=device == \"cuda\")\n    for ep in range(cfg.cnn_epochs):\n        model.train()\n        t0, tl = time.time(), 0.0\n        for x, y, hb, has in dl:\n            x, y, hb, has = x.to(device), y.to(device), hb.to(device), has.to(device)\n            with torch.autocast(device_type=device, enabled=device == \"cuda\"):\n                lc, lh = model(x)\n                loss = F.binary_cross_entropy_with_logits(lc.float(), y, pos_weight=pos_weight)\n                if has.sum() > 0:\n                    lhb = (F.smooth_l1_loss(lh.float(), hb, reduction=\"none\") * has).sum() / has.sum()\n                    loss = loss + cfg.hb_loss_weight * lhb\n            opt.zero_grad()\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            nn.utils.clip_grad_norm_(model.parameters(), 2.0)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            tl += loss.item()\n        if (ep + 1) % 5 == 0 or ep == 0 or ep + 1 == cfg.cnn_epochs:\n            print(f\"    {log_prefix}epoch {ep + 1:02d}/{cfg.cnn_epochs} loss {tl / max(1, len(dl)):.3f} ({time.time() - t0:.0f}s)\")\n    return model\n\n\n@torch.no_grad()\ndef predict_cnn(model, paths, size, device, batch=64):\n    model.eval()\n    ds = RoiDS(paths, np.zeros(len(paths)), np.full(len(paths), np.nan), size, False)\n    dl = DataLoader(ds, batch_size=batch, shuffle=False)\n    probs, hbs = [], []\n    for x, *_ in dl:\n        x = x.to(device)\n        c1, h1 = model(x)\n        c2, h2 = model(torch.flip(x, [3]))                # TTA flip horizontal\n        probs.append(torch.sigmoid((c1 + c2) / 2).float().cpu().numpy())\n        hbs.append(((h1 + h2) / 2).float().cpu().numpy() * HB_SD + HB_MU)\n    return np.concatenate(probs), np.concatenate(hbs)\n\n\n@torch.no_grad()\ndef predict_cnn_arrays(model, imgs, size, device):\n    \"\"\"Seperti predict_cnn tetapi untuk array RGB (dipakai aplikasi).\"\"\"\n    model.eval()\n    x = torch.stack([to_tensor(cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)) for a in imgs]).to(device)\n    c1, h1 = model(x)\n    c2, h2 = model(torch.flip(x, [3]))\n    return (torch.sigmoid((c1 + c2) / 2).float().cpu().numpy(),\n            ((h1 + h2) / 2).float().cpu().numpy() * HB_SD + HB_MU)\n\n\n# ---------------------------------------------------------------- DINOv2\ndef build_dino(name, pretrained=True, size=224):\n    import timm\n    m = timm.create_model(name, pretrained=pretrained, num_classes=0, img_size=size)\n    return m.eval()\n\n\n@torch.no_grad()\ndef dino_embed(model, imgs_or_paths, size, device, batch=64):\n    \"\"\"Embedding beku + TTA flip. Menerima daftar path ROI atau array RGB.\"\"\"\n    model = model.to(device)\n    out = []\n    for i in range(0, len(imgs_or_paths), batch):\n        chunk = imgs_or_paths[i:i + batch]\n        arr = [load_roi(p, size) if isinstance(p, str) else cv2.resize(p, (size, size), interpolation=cv2.INTER_AREA)\n               for p in chunk]\n        x = torch.stack([to_tensor(a) for a in arr]).to(device)\n        with torch.autocast(device_type=device, enabled=device == \"cuda\"):\n            e = (model(x) + model(torch.flip(x, [3]))) / 2\n        out.append(F.normalize(e.float(), dim=1).cpu().numpy())\n    return np.concatenate(out)\n\n\n# ---------------------------------------------------------- model sklearn\ndef make_clf(groups_inner=None):\n    from sklearn.linear_model import LogisticRegressionCV\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    return make_pipeline(StandardScaler(), LogisticRegressionCV(\n        Cs=[0.003, 0.01, 0.03, 0.1, 0.3, 1.0], cv=groups_inner or 3, scoring=\"roc_auc\",\n        class_weight=\"balanced\", max_iter=5000))\n\n\ndef make_reg():\n    from sklearn.linear_model import RidgeCV\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12)))\n\n\ndef make_rf_clf():\n    from sklearn.ensemble import RandomForestClassifier\n    return RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight=\"balanced_subsample\",\n                                  random_state=0, n_jobs=-1)\n\n\ndef make_rf_reg():\n    from sklearn.ensemble import RandomForestRegressor\n    return RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=0, n_jobs=-1)\n", "train_eval.py": "\"\"\"Langkah 7: validasi silang 5-fold BERBASIS PASIEN untuk 3 model + ensemble.\nSemua angka di laporan adalah prediksi out-of-fold (gambar tidak pernah dinilai oleh\nmodel yang melihatnya saat training), ditambah uji leave-one-source-out untuk\nmengukur seberapa baik model bertahan di dataset/negara yang belum pernah dilihat.\n\nPakai:  python anemialens/train_eval.py /content/work            (lengkap)\n        python anemialens/train_eval.py /content/work --no-cnn   (cepat, tanpa CNN)\nHasil:  work/oof.csv, work/thresholds.json, work/report.md, work/roc.png, work/hb_scatter.png\n\"\"\"\nimport json\nimport sys\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import roc_auc_score, roc_curve\nfrom sklearn.model_selection import GroupKFold, StratifiedGroupKFold\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import COLOR_FEATURE_NAMES, Config, seed_everything  # noqa: E402\nfrom anemialens.models import (build_dino, dino_embed, make_clf, make_reg, make_rf_clf,  # noqa: E402\n                               make_rf_reg, predict_cnn, train_cnn)\n\n\nwarnings.filterwarnings(\"ignore\")\n\n\ndef inner_splits(groups, y, k=3):\n    k = min(k, len(np.unique(groups)))\n    return list(GroupKFold(n_splits=k).split(np.zeros(len(y)), y, groups))\n\n\ndef fit_predict_sklearn(Xtr, ytr, gtr, hbtr, Xte, kind=\"linear\"):\n    clf = make_rf_clf() if kind == \"rf\" else make_clf(inner_splits(gtr, ytr))\n    clf.fit(Xtr, ytr)\n    p = clf.predict_proba(Xte)[:, 1]\n    hb = np.full(len(Xte), np.nan)\n    m = ~np.isnan(hbtr)\n    if m.sum() >= 20:\n        reg = (make_rf_reg() if kind == \"rf\" else make_reg()).fit(Xtr[m], hbtr[m])\n        hb = reg.predict(Xte)\n    return p, hb\n\n\ndef thr_for_sens(y, p, target):\n    fpr, tpr, thr = roc_curve(y, p)\n    ok = np.where(tpr >= target)[0]\n    return float(thr[ok[0]]) if len(ok) else 0.5\n\n\ndef thr_for_spec(y, p, target=0.90):\n    fpr, tpr, thr = roc_curve(y, p)\n    ok = np.where(1 - fpr >= target)[0]\n    return float(thr[ok[-1]]) if len(ok) else 0.5\n\n\ndef metrics(y, p, thr, hb_true=None, hb_pred=None):\n    out = {\"n\": int(len(y)), \"anemia\": int(y.sum())}\n    if len(np.unique(y)) == 2:\n        out[\"auc\"] = roc_auc_score(y, p)\n    pred = (p >= thr).astype(int)\n    tp, tn = ((pred == 1) & (y == 1)).sum(), ((pred == 0) & (y == 0)).sum()\n    fp, fn = ((pred == 1) & (y == 0)).sum(), ((pred == 0) & (y == 1)).sum()\n    out.update(sens=tp / max(1, tp + fn), spec=tn / max(1, tn + fp), ppv=tp / max(1, tp + fp),\n               npv=tn / max(1, tn + fn), acc=(tp + tn) / len(y))\n    out[\"bal_acc\"] = (out[\"sens\"] + out[\"spec\"]) / 2\n    if hb_true is not None and hb_pred is not None:\n        m = ~np.isnan(hb_true) & ~np.isnan(hb_pred)\n        if m.sum() >= 5:\n            e = hb_pred[m] - hb_true[m]\n            out.update(hb_n=int(m.sum()), hb_mae=float(np.abs(e).mean()),\n                       hb_r=float(np.corrcoef(hb_true[m], hb_pred[m])[0, 1]),\n                       hb_within1=float((np.abs(e) <= 1).mean()), hb_within2=float((np.abs(e) <= 2).mean()))\n    return out\n\n\ndef md(d, index=True):\n    try:\n        return d.to_markdown(index=index, floatfmt=\".3f\")\n    except ImportError:\n        return d.to_string(index=index)\n\n\ndef bootstrap_auc_ci(y, p, groups, n=1000, seed=0):\n    rng = np.random.default_rng(seed)\n    ug = np.unique(groups)\n    idx_by_g = {g: np.where(groups == g)[0] for g in ug}\n    aucs = []\n    for _ in range(n):\n        s = np.concatenate([idx_by_g[g] for g in rng.choice(ug, len(ug), replace=True)])\n        if len(np.unique(y[s])) == 2:\n            aucs.append(roc_auc_score(y[s], p[s]))\n    return np.percentile(aucs, [2.5, 97.5]) if aucs else (np.nan, np.nan)\n\n\ndef main(work_dir, use_cnn=True):\n    work = Path(work_dir)\n    cfg = Config.load(work / \"config.json\") if (work / \"config.json\").exists() else Config(work_dir=str(work))\n    seed_everything(cfg.seed)\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    df = pd.read_csv(work / \"features.csv\", keep_default_na=False, na_values=[\"\"])\n    df[\"mask_path\"] = df.mask_path.fillna(\"\")\n    y = df.label.values.astype(int)\n    hb = pd.to_numeric(df.hb, errors=\"coerce\").values.astype(float)\n    groups = df.patient_id.astype(str).values\n    src = df.source.astype(str).values\n    Xc = df[COLOR_FEATURE_NAMES].values.astype(np.float32)\n    print(f\"Data: {len(df)} foto, {len(np.unique(groups))} pasien, anemia {y.mean():.1%}, punya Hb {np.isfinite(hb).sum()}\")\n\n    # --------------------------------------------------- embedding DINOv2 (sekali)\n    emb_path = work / \"dino_emb.npy\"\n    if emb_path.exists() and np.load(emb_path).shape[0] == len(df):\n        Xd = np.load(emb_path)\n    else:\n        print(\"Menghitung embedding DINOv2...\")\n        Xd = dino_embed(build_dino(cfg.dino_backbone, cfg.pretrained, cfg.img_size), df.roi_path.tolist(),\n                        cfg.img_size, device)\n        np.save(emb_path, Xd)\n\n    # --------------------------------------------------- 5-fold berbasis pasien\n    strat = np.array([f\"{a}_{b}\" for a, b in zip(y, src)])\n    k = min(cfg.n_folds, len(np.unique(groups)))\n    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=cfg.seed)\n    names = [\"color\", \"color_rf\", \"dino\", \"cnn\"]\n    oof = {m: np.full(len(df), np.nan) for m in names}\n    oof_hb = {m: np.full(len(df), np.nan) for m in names}\n    fold_id = np.full(len(df), -1)\n    for f, (tr, te) in enumerate(sgkf.split(Xc, strat, groups)):\n        fold_id[te] = f\n        print(f\"\\nFold {f + 1}/{k}: latih {len(tr)} | uji {len(te)}\")\n        oof[\"color\"][te], oof_hb[\"color\"][te] = fit_predict_sklearn(Xc[tr], y[tr], groups[tr], hb[tr], Xc[te])\n        oof[\"color_rf\"][te], oof_hb[\"color_rf\"][te] = fit_predict_sklearn(Xc[tr], y[tr], groups[tr], hb[tr], Xc[te], \"rf\")\n        oof[\"dino\"][te], oof_hb[\"dino\"][te] = fit_predict_sklearn(Xd[tr], y[tr], groups[tr], hb[tr], Xd[te])\n        print(f\"  AUC fold  warna={roc_auc_score(y[te], oof['color'][te]):.3f}  \"\n              f\"warna_rf={roc_auc_score(y[te], oof['color_rf'][te]):.3f}  \"\n              f\"dino={roc_auc_score(y[te], oof['dino'][te]):.3f}\" if len(np.unique(y[te])) == 2 else \"\")\n        if use_cnn:\n            model = train_cnn(cfg, df.roi_path.values[tr], y[tr], hb[tr], device, log_prefix=f\"[cnn f{f + 1}] \")\n            oof[\"cnn\"][te], oof_hb[\"cnn\"][te] = predict_cnn(model, df.roi_path.values[te].tolist(), cfg.img_size, device)\n            if len(np.unique(y[te])) == 2:\n                print(f\"  AUC fold  cnn={roc_auc_score(y[te], oof['cnn'][te]):.3f}\")\n            del model\n            if device == \"cuda\":\n                torch.cuda.empty_cache()\n\n    avail = [m for m in oof if not np.isnan(oof[m]).all()]\n    aucs = {m: roc_auc_score(y, oof[m]) for m in avail}\n    best = max(aucs.values())\n    # Anggota ensemble: model yang jelas berguna (AUC >= 0.60 dan tidak jauh dari terbaik).\n    used = [m for m in avail if aucs[m] >= 0.60 and aucs[m] >= best - 0.08] or [max(aucs, key=aucs.get)]\n    print(\"\\nAUC out-of-fold:\", {m: round(a, 3) for m, a in aucs.items()}, \"-> anggota ensemble:\", used)\n    oof[\"ensemble\"] = np.mean([oof[m] for m in used], axis=0)\n    hb_stack = np.array([oof_hb[m] for m in used])\n    oof_hb[\"ensemble\"] = np.nanmean(hb_stack, axis=0) if np.isfinite(hb_stack).any() else np.full(len(df), np.nan)\n\n    # --------------------------------------------------- ambang & metrik\n    thr_s = thr_for_sens(y, oof[\"ensemble\"], cfg.target_sensitivity)\n    thr_p = max(thr_for_spec(y, oof[\"ensemble\"], 0.90), thr_s)\n    rows = []\n    for m in avail + [\"ensemble\"]:\n        t = thr_for_sens(y, oof[m], cfg.target_sensitivity)\n        r = metrics(y, oof[m], t, hb, oof_hb[m])\n        r[\"model\"] = m\n        rows.append(r)\n    res = pd.DataFrame(rows).set_index(\"model\")\n    lo, hi = bootstrap_auc_ci(y, oof[\"ensemble\"], groups)\n    per_src = []\n    for s in np.unique(src):\n        mm = src == s\n        r = metrics(y[mm], oof[\"ensemble\"][mm], thr_s, hb[mm], oof_hb[\"ensemble\"][mm])\n        r[\"source\"] = s\n        per_src.append(r)\n    per_src = pd.DataFrame(per_src).set_index(\"source\")\n\n    # --------------------------------------------------- leave-one-source-out\n    loso = []\n    if len(np.unique(src)) > 1:\n        print(\"\\nUji leave-one-source-out (latih di sumber lain, uji di satu sumber)...\")\n        for s in np.unique(src):\n            te, tr = src == s, src != s\n            if len(np.unique(y[te])) < 2 or len(np.unique(y[tr])) < 2:\n                continue\n            row = {\"source_uji\": s, \"n\": int(te.sum())}\n            ps = []\n            for name, X, kind in ((\"color\", Xc, \"linear\"), (\"color_rf\", Xc, \"rf\"), (\"dino\", Xd, \"linear\")):\n                p, _ = fit_predict_sklearn(X[tr], y[tr], groups[tr], hb[tr], X[te], kind)\n                row[f\"auc_{name}\"] = roc_auc_score(y[te], p)\n                ps.append(p)\n            row[\"auc_rata2\"] = roc_auc_score(y[te], np.mean(ps, axis=0))\n            loso.append(row)\n    loso = pd.DataFrame(loso)\n\n    # --------------------------------------------------- simpan\n    out = df[[\"image_path\", \"source\", \"patient_id\", \"label\", \"hb\"]].copy()\n    out[\"fold\"] = fold_id\n    for m in oof:\n        out[f\"p_{m}\"] = oof[m]\n        out[f\"hb_{m}\"] = oof_hb.get(m, np.nan)\n    out.to_csv(work / \"oof.csv\", index=False)\n    thresholds = {\"thr_sensitive\": thr_s, \"thr_specific\": thr_p, \"target_sensitivity\": cfg.target_sensitivity,\n                  \"models\": used, \"auc_ensemble\": float(res.loc[\"ensemble\", \"auc\"]),\n                  \"auc_ci95\": [float(lo), float(hi)],\n                  \"hb_mae\": float(res.loc[\"ensemble\", \"hb_mae\"]) if \"hb_mae\" in res.columns\n                  and pd.notna(res.loc[\"ensemble\", \"hb_mae\"]) else None}\n    (work / \"thresholds.json\").write_text(json.dumps(thresholds, indent=2))\n\n    pd.set_option(\"display.float_format\", lambda v: f\"{v:.3f}\")\n    cols = [c for c in [\"n\", \"anemia\", \"auc\", \"sens\", \"spec\", \"ppv\", \"npv\", \"bal_acc\",\n                        \"hb_mae\", \"hb_r\", \"hb_within1\", \"hb_within2\"] if c in res.columns]\n    report = [\n        \"# Laporan validasi AnemiaLens (out-of-fold, split per pasien)\\n\",\n        f\"Data: {len(df)} foto, {len(np.unique(groups))} pasien, prevalensi anemia {y.mean():.1%}.\\n\",\n        f\"Anggota ensemble: {', '.join(used)} (dipilih dari AUC out-of-fold; model lain hanya dilaporkan).\\n\",\n        f\"AUC ensemble: **{res.loc['ensemble', 'auc']:.3f}** (95% CI bootstrap per pasien {lo:.3f}–{hi:.3f}).\\n\",\n        f\"Ambang skrining (sensitivitas ≥ {cfg.target_sensitivity:.0%}): {thr_s:.3f}; \"\n        f\"ambang risiko tinggi (spesifisitas ≥ 90%): {thr_p:.3f}.\\n\",\n        \"\\n## Per model\\n\", md(res[cols]), \"\\n\\n## Ensemble per sumber data\\n\",\n        md(per_src[[c for c in cols if c in per_src.columns]]),\n    ]\n    if len(loso):\n        report += [\"\\n\\n## Leave-one-source-out (proksi validasi eksternal)\\n\", md(loso, index=False)]\n    (work / \"report.md\").write_text(\"\\n\".join(report))\n    print(\"\\n\" + \"\\n\".join(report))\n\n    try:\n        import matplotlib\n        matplotlib.use(\"Agg\")\n        import matplotlib.pyplot as plt\n        fig, ax = plt.subplots(figsize=(5, 5))\n        for m in avail + [\"ensemble\"]:\n            fpr, tpr, _ = roc_curve(y, oof[m])\n            ax.plot(fpr, tpr, lw=2.5 if m == \"ensemble\" else 1.2, label=f\"{m} (AUC {roc_auc_score(y, oof[m]):.3f})\")\n        ax.plot([0, 1], [0, 1], \"k--\", lw=0.8)\n        ax.set(xlabel=\"1 - spesifisitas\", ylabel=\"sensitivitas\", title=\"ROC out-of-fold\")\n        ax.legend(loc=\"lower right\", fontsize=8)\n        fig.tight_layout()\n        fig.savefig(work / \"roc.png\", dpi=150)\n        m = np.isfinite(hb) & np.isfinite(oof_hb[\"ensemble\"])\n        if m.sum() >= 5:\n            fig, ax = plt.subplots(figsize=(5, 5))\n            ax.scatter(hb[m], oof_hb[\"ensemble\"][m], s=10, alpha=0.6)\n            lims = [min(hb[m].min(), oof_hb[\"ensemble\"][m].min()) - 0.5, max(hb[m].max(), oof_hb[\"ensemble\"][m].max()) + 0.5]\n            ax.plot(lims, lims, \"k--\", lw=0.8)\n            ax.set(xlabel=\"Hb lab (g/dL)\", ylabel=\"Hb prediksi (g/dL)\", title=\"Estimasi Hb out-of-fold\")\n            fig.tight_layout()\n            fig.savefig(work / \"hb_scatter.png\", dpi=150)\n    except Exception as e:  # noqa: BLE001\n        print(\"Plot dilewati:\", e)\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith(\"--\") else \"/content/work\",\n         use_cnn=\"--no-cnn\" not in sys.argv)\n", "train_final.py": "\"\"\"Langkah 8: latih model final memakai SEMUA data (ambang dari validasi silang),\nlalu simpan semua artefak yang dibutuhkan aplikasi ke work/model/.\n\nPakai:  python anemialens/train_final.py /content/work [--no-cnn]\n\"\"\"\nimport json\nimport shutil\nimport sys\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import COLOR_FEATURE_NAMES, Config, seed_everything  # noqa: E402\nfrom anemialens.models import make_clf, make_reg, make_rf_clf, make_rf_reg, train_cnn  # noqa: E402\nfrom anemialens.train_eval import inner_splits  # noqa: E402\n\n\ndef main(work_dir, use_cnn=True):\n    work = Path(work_dir)\n    cfg = Config.load(work / \"config.json\") if (work / \"config.json\").exists() else Config(work_dir=str(work))\n    seed_everything(cfg.seed)\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    out = work / \"model\"\n    out.mkdir(exist_ok=True)\n    df = pd.read_csv(work / \"features.csv\", keep_default_na=False, na_values=[\"\"])\n    y = df.label.values.astype(int)\n    hb = pd.to_numeric(df.hb, errors=\"coerce\").values.astype(float)\n    g = df.patient_id.astype(str).values\n    Xc = df[COLOR_FEATURE_NAMES].values.astype(np.float32)\n    Xd = np.load(work / \"dino_emb.npy\")\n    thr = json.loads((work / \"thresholds.json\").read_text())\n    if use_cnn and \"cnn\" not in thr[\"models\"]:\n        print(\"Info: CNN bukan anggota ensemble (AUC validasi lebih rendah / tidak dilatih), jadi tidak dilatih ulang.\")\n        use_cnn = False\n    print(\"Anggota ensemble:\", thr[\"models\"])\n    m = ~np.isnan(hb)\n    for name, X, kind in ((\"color\", Xc, \"linear\"), (\"color_rf\", Xc, \"rf\"), (\"dino\", Xd, \"linear\")):\n        clf = make_rf_clf() if kind == \"rf\" else make_clf(inner_splits(g, y))\n        joblib.dump(clf.fit(X, y), out / f\"{name}_clf.joblib\")\n        if m.sum() >= 20:\n            reg = make_rf_reg() if kind == \"rf\" else make_reg()\n            joblib.dump(reg.fit(X[m], hb[m]), out / f\"{name}_hb.joblib\")\n        print(f\"  model {name} tersimpan\")\n    if use_cnn:\n        print(\"Melatih CNN final pada semua data...\")\n        model = train_cnn(cfg, df.roi_path.values, y, hb, device, log_prefix=\"[cnn final] \")\n        torch.save(model.state_dict(), out / \"cnn.pt\")\n    if (work / \"seg_unet.pt\").exists():\n        shutil.copy(work / \"seg_unet.pt\", out / \"seg_unet.pt\")\n    (out / \"thresholds.json\").write_text(json.dumps(thr, indent=2))\n    cfg.save(out / \"config.json\")\n    for f in (\"report.md\", \"roc.png\", \"hb_scatter.png\", \"seg_metrics.txt\", \"quality_ref.json\"):\n        if (work / f).exists():\n            shutil.copy(work / f, out / f)\n    from anemialens.roi_guard import build_ref\n    build_ref(work, out)\n    print(f\"Artefak model siap di: {out}\")\n    print(sorted(p.name for p in out.iterdir()))\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith(\"--\") else \"/content/work\",\n         use_cnn=\"--no-cnn\" not in sys.argv)\n", "predict.py": "\"\"\"Inferensi AnemiaLens untuk satu foto.\n\n    from anemialens.predict import AnemiaLens\n    lens = AnemiaLens(\"/content/work/model\")\n    hasil = lens.analyze(rgb_array, use_card=True)\n\nAlur: cek kualitas -> (opsional) koreksi warna dengan kartu putih -> segmentasi\nkonjungtiva -> crop -> 3 model -> ensemble -> tingkat risiko + estimasi Hb.\nPakai CLI:  python anemialens/predict.py /content/work/model foto.jpg\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport joblib\nimport numpy as np\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import (Config, color_features, crop_roi, letterbox, quality_check,  # noqa: E402\n                               read_rgb, risk_tier, white_balance_from_patch)\nfrom anemialens.models import MultiTaskNet, build_dino, dino_embed, predict_cnn_arrays  # noqa: E402\nfrom anemialens.roi_guard import check as roi_check, manual_mask, refine_user_mask  # noqa: E402\n\nBLOCKING = {\"buram\", \"terlalu_gelap\", \"terlalu_terang\", \"pantulan_cahaya\",\n            \"konjungtiva_terlalu_kecil\", \"konjungtiva_tidak_terdeteksi\"}\nADVICE = {\n    \"buram\": \"Foto buram. Pegang HP stabil dan ketuk layar untuk fokus pada mata.\",\n    \"terlalu_gelap\": \"Foto terlalu gelap. Foto di dekat jendela atau lampu putih, jangan pakai flash.\",\n    \"terlalu_terang\": \"Foto terlalu terang. Jauhi cahaya langsung atau matikan flash.\",\n    \"pantulan_cahaya\": \"Ada pantulan cahaya di kelopak mata. Ubah sudut HP sedikit.\",\n    \"konjungtiva_terlalu_kecil\": \"Bagian dalam kelopak bawah terlalu kecil. Dekatkan kamera dan tarik kelopak bawah ke bawah.\",\n    \"konjungtiva_tidak_terdeteksi\": \"AI belum menemukan bagian dalam kelopak bawah. Tarik kelopak bawah lebih lebar lalu foto ulang, \"\n                                    \"atau lingkari sendiri bagian dalam kelopak (yang basah, merah muda atau pucat).\",\n    \"bentuk_roi_tidak_wajar\": \"Area yang terdeteksi tidak berbentuk seperti bagian dalam kelopak bawah. Tarik kelopak bawah \"\n                              \"sampai bagian dalam yang merah muda terlihat jelas, foto dari dekat (mata memenuhi layar), \"\n                              \"atau lingkari sendiri bagian dalam kelopak.\",\n    \"roi_tidak_kemerahan\": \"Area yang terdeteksi lebih mirip kulit atau bagian putih mata daripada konjungtiva. \"\n                           \"Tarik kelopak bawah lebih jelas, atau lingkari tepat bagian dalam kelopak (bukan kulit di bawahnya).\",\n    \"warna_di_luar_data_latih\": \"Warna atau pencahayaan foto terlalu berbeda dari data latih, jadi hasilnya tidak bisa \"\n                                \"dipercaya. Foto ulang di cahaya putih yang merata, tanpa flash dan tanpa filter.\",\n}\nGUARD_WARN = (\"Warna area yang dilingkari di luar jangkauan data latih: bisa karena sangat pucat, atau area itu bukan \"\n              \"konjungtiva. Hasil kurang pasti, sebaiknya langsung cek hemoglobin.\")\nDISCLAIMER = (\"Hasil ini adalah skrining awal berbasis foto, BUKAN diagnosis. \"\n              \"Pastikan dengan pemeriksaan hemoglobin (cek darah) di puskesmas, klinik, atau laboratorium.\")\n\n\ndef auto_white_patch(rgb):\n    \"\"\"Cari kartu putih/abu: area besar, saturasi rendah, cukup terang.\"\"\"\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    cand = ((hsv[:, :, 1] < 35) & (hsv[:, :, 2] > 150)).astype(np.uint8)\n    cand = cv2.morphologyEx(cand, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))\n    n, lab, st, _ = cv2.connectedComponentsWithStats(cand, 8)\n    total = rgb.shape[0] * rgb.shape[1]\n    best = None\n    for k in range(1, n):\n        x, y, w, h, a = st[k]\n        rect = a / max(1, w * h)                 # kartu = persegi panjang penuh; sklera = lonjong\n        if 0.01 * total <= a <= 0.4 * total and rect >= 0.85 and 0.25 <= w / max(1, h) <= 4:\n            if best is None or a > best[4]:\n                best = (x, y, w, h, a)\n    if best is None:\n        return None\n    x, y, w, h, _ = best\n    sx, sy = int(w * 0.2), int(h * 0.2)          # ambil bagian tengah kartu\n    return (x + sx, y + sy, x + w - sx, y + h - sy)\n\n\nclass AnemiaLens:\n    def __init__(self, model_dir):\n        d = Path(model_dir)\n        self.cfg = Config.load(d / \"config.json\")\n        self.thr = json.loads((d / \"thresholds.json\").read_text())\n        self.device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        self.seg = None\n        if (d / \"seg_unet.pt\").exists():\n            from anemialens.train_seg import build_unet\n            self.seg = build_unet(False).to(self.device)\n            self.seg.load_state_dict(torch.load(d / \"seg_unet.pt\", map_location=self.device))\n            self.seg.eval()\n        names = [n for n in (\"color\", \"color_rf\", \"dino\") if (d / f\"{n}_clf.joblib\").exists()]\n        self.sk = {n: joblib.load(d / f\"{n}_clf.joblib\") for n in names}\n        self.hbm = {n: joblib.load(d / f\"{n}_hb.joblib\") for n in names if (d / f\"{n}_hb.joblib\").exists()}\n        # DINOv2 hanya dimuat bila termasuk ensemble (hemat RAM & tidak perlu unduh dari HF Hub)\n        if \"dino\" not in self.thr[\"models\"]:\n            self.sk.pop(\"dino\", None)\n            self.hbm.pop(\"dino\", None)\n        self.dino = (build_dino(self.cfg.dino_backbone, self.cfg.pretrained, self.cfg.img_size)\n                     if \"dino\" in self.sk else None)\n        self.cnn = None\n        if \"cnn\" in self.thr[\"models\"] and (d / \"cnn.pt\").exists():\n            self.cnn = MultiTaskNet(self.cfg.cnn_backbone, pretrained=False).to(self.device)\n            self.cnn.load_state_dict(torch.load(d / \"cnn.pt\", map_location=self.device))\n            self.cnn.eval()\n        self.hb_mae = self.thr.get(\"hb_mae\")\n        qr = d / \"quality_ref.json\"\n        self.qref = json.loads(qr.read_text()) if qr.exists() else None\n        rr = d / \"roi_ref.json\"\n        self.rref = json.loads(rr.read_text()) if rr.exists() else None\n        if self.rref is None:\n            print(\"PERINGATAN: roi_ref.json belum ada. Jalankan roi_guard.py agar foto non-konjungtiva ditolak.\")\n\n    def segment(self, rgb):\n        if self.seg is None:\n            return None, \"full\"\n        from anemialens.train_seg import predict_mask\n        m = predict_mask(self.seg, rgb, self.cfg.seg_size, self.device)\n        a = m.mean()\n        if a >= 0.45:\n            return None, \"full\"\n        if a < 0.005:\n            return np.zeros_like(m), \"none\"\n        return m, \"pred\"\n\n    def analyze(self, rgb, use_card=False, force=False, mask=None, manual_crop=False):\n        card = None\n        if use_card:\n            card = auto_white_patch(rgb)\n            if card is not None:\n                rgb = white_balance_from_patch(rgb, card)\n        if manual_crop and mask is not None:  # pengguna melingkari konjungtiva sendiri (lasso)\n            mask, mode = refine_user_mask(rgb, mask), \"manual\"\n        elif manual_crop:                     # pengguna hanya crop kotak ke sekitar konjungtiva\n            mask, mode = manual_mask(rgb), \"manual\"\n        elif mask is None:\n            mask, mode = self.segment(rgb)\n        else:\n            mode = \"pred\" if mask.any() else \"full\"\n        q = quality_check(rgb, mask if mode not in (\"full\",) else None, self.qref)\n        issues = [i for i in q[\"issues\"] if i in BLOCKING]\n        overlay = rgb.copy()\n        if mask is not None and mask.any():\n            cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n            cv2.drawContours(overlay, cnts, -1, (0, 255, 120), max(2, rgb.shape[1] // 300))\n        if card is not None:\n            cv2.rectangle(overlay, card[:2], card[2:], (80, 160, 255), max(2, rgb.shape[1] // 300))\n        base = {\"quality\": q, \"roi_mode\": mode, \"card_used\": card is not None, \"overlay\": overlay,\n                \"disclaimer\": DISCLAIMER}\n        if issues and not force:\n            return {**base, \"status\": \"foto_tidak_layak\", \"advice\": [ADVICE[i] for i in issues]}\n        roi = crop_roi(rgb, mask if mode in (\"pred\", \"manual\") else None)\n        roi_std = letterbox(roi, 448)                       # sama dengan pra-proses saat training\n        xc = color_features(roi)[None]\n        guard = roi_check(self.rref, mask, mode, xc)\n        guard_warn = []\n        if guard and mode == \"manual\":\n            # Area ditandai sendiri oleh pengguna: jangan ditolak, karena konjungtiva yang sangat pucat\n            # (anemia berat) bisa jatuh di luar jangkauan warna data latih. Beri peringatan saja.\n            guard_warn = [GUARD_WARN]\n        elif guard and not force:\n            return {**base, \"status\": \"foto_tidak_layak\", \"advice\": [ADVICE[g] for g in guard], \"guard\": guard}\n        feats = {\"color\": xc, \"color_rf\": xc}\n        if self.dino is not None:\n            feats[\"dino\"] = dino_embed(self.dino, [roi_std], self.cfg.img_size, self.device)\n        probs = {n: float(m.predict_proba(feats[n])[0, 1]) for n, m in self.sk.items()}\n        hbs = [float(m.predict(feats[n])[0]) for n, m in self.hbm.items() if n in self.thr[\"models\"]]\n        if self.cnn is not None:\n            p, h = predict_cnn_arrays(self.cnn, [roi_std], self.cfg.img_size, self.device)\n            probs[\"cnn\"] = float(p[0])\n            hbs.append(float(h[0]))\n        member = [probs[m] for m in self.thr[\"models\"] if m in probs]\n        prob = float(np.mean(member))\n        disagree = len(member) > 1 and (max(member) - min(member)) > 0.5\n        hbs = [float(np.clip(h, 4.0, 20.0)) for h in hbs]\n        tier = risk_tier(prob, self.thr[\"thr_sensitive\"], self.thr[\"thr_specific\"])\n        hb = float(np.mean(hbs)) if hbs else None\n        msg = {\n            \"tinggi\": \"Risiko anemia TINGGI. Segera periksa hemoglobin di puskesmas/klinik dalam beberapa hari ini.\",\n            \"sedang\": \"Ada tanda kemungkinan anemia. Disarankan cek hemoglobin untuk memastikan.\",\n            \"rendah\": \"Tidak tampak tanda pucat yang jelas. Tetap cek Hb bila lemas, pusing, atau sesak berlanjut.\",\n        }[tier]\n        return {**base, \"status\": \"ok\", \"probability\": prob, \"risk_tier\": tier, \"message\": msg,\n                \"hb_estimate\": hb, \"hb_mae\": self.hb_mae, \"per_model\": probs,\n                \"quality_warnings\": [ADVICE[i] for i in q[\"issues\"] if i in ADVICE]\n                + ([\"Model-model tidak sepakat untuk foto ini, jadi hasilnya kurang yakin. Disarankan foto ulang.\"]\n                   if disagree else []) + guard_warn, \"guard\": guard}\n\n\nif __name__ == \"__main__\":\n    lens = AnemiaLens(sys.argv[1])\n    r = lens.analyze(read_rgb(sys.argv[2]), use_card=\"--card\" in sys.argv)\n    r.pop(\"overlay\")\n    print(json.dumps(r, indent=2, ensure_ascii=False, default=str))\n", "show_samples.py": "\"\"\"Cek visual: grid contoh ROI per sumber & kelas, dan contoh mask hasil segmentasi.\nPakai:  python anemialens/show_samples.py /content/work\nHasil:  work/samples_roi.png, work/samples_mask.png\n\"\"\"\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import read_mask, read_rgb  # noqa: E402\n\nT = 150\n\n\ndef tile(img):\n    return cv2.resize(img, (T, T), interpolation=cv2.INTER_AREA)\n\n\ndef label_bar(text, w):\n    bar = np.full((22, w, 3), 255, np.uint8)\n    cv2.putText(bar, text[:60], (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (20, 20, 20), 1, cv2.LINE_AA)\n    return bar\n\n\ndef main(work_dir, per=6):\n    work = Path(work_dir)\n    rng = np.random.default_rng(0)\n    if (work / \"features.csv\").exists():\n        df = pd.read_csv(work / \"features.csv\", keep_default_na=False)\n        rows = []\n        for (src, lab), g in df.groupby([\"source\", \"label\"]):\n            pick = g.sample(min(per, len(g)), random_state=0)\n            ims = [tile(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)) for p in pick.roi_path]\n            ims += [np.full((T, T, 3), 255, np.uint8)] * (per - len(ims))\n            rows += [label_bar(f\"{src} | {'ANEMIA' if lab == 1 else 'normal'} | n={len(g)}\", per * T), np.hstack(ims)]\n        cv2.imwrite(str(work / \"samples_roi.png\"), cv2.cvtColor(np.vstack(rows), cv2.COLOR_RGB2BGR))\n        print(\"Tersimpan:\", work / \"samples_roi.png\")\n    df = pd.read_csv(work / \"manifest.csv\", keep_default_na=False)\n    if \"roi_mode\" in df:\n        rows = []\n        for mode in (\"gt\", \"pred\", \"alpha\"):\n            g = df[df.roi_mode == mode]\n            if not len(g):\n                continue\n            pick = g.iloc[rng.choice(len(g), min(per, len(g)), replace=False)]\n            ims = []\n            for _, r in pick.iterrows():\n                img = read_rgb(r.image_path)\n                m = read_mask(r.roi_mask, img.shape)\n                ov = img.copy()\n                ov[m > 0] = (0.5 * ov[m > 0] + 0.5 * np.array([0, 255, 120])).astype(np.uint8)\n                ims.append(tile(ov))\n            ims += [np.full((T, T, 3), 255, np.uint8)] * (per - len(ims))\n            rows += [label_bar({\"gt\": \"mask asli (dataset)\", \"pred\": \"mask prediksi U-Net\", \"alpha\": \"sudah ter-crop (latar transparan)\"}[mode], per * T), np.hstack(ims)]\n        if rows:\n            cv2.imwrite(str(work / \"samples_mask.png\"), cv2.cvtColor(np.vstack(rows), cv2.COLOR_RGB2BGR))\n            print(\"Tersimpan:\", work / \"samples_mask.png\")\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1] if len(sys.argv) > 1 else \"/content/work\")\n", "roi_guard.py": "\"\"\"Pengaman ROI: memastikan area yang dinilai benar-benar konjungtiva, bukan kulit/sklera.\n\n1. build_ref(): dari data latih, simpan statistik \"seperti apa konjungtiva yang sah\":\n   - bentuk mask (luas relatif, kelonjongan) dari mask manual dataset,\n   - warna (a* minimum) dan jarak Mahalanobis fitur warna (deteksi out-of-distribution).\n2. check(): menolak ROI yang bentuk/warnanya di luar jangkauan data latih.\n3. manual_mask(): untuk foto yang sudah di-crop manual oleh pengguna, ambil piksel\n   kemerahan (CIELAB a* tinggi) sebagai konjungtiva.\n\nPakai:  python anemialens/roi_guard.py /content/work /content/work/model\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.common import COLOR_FEATURE_NAMES, read_mask  # noqa: E402\n\n\nAREA_MIN_ABS = 0.002\n\n\ndef mask_shape(mask):\n    \"\"\"(luas relatif, kelonjongan) dari komponen terbesar.\"\"\"\n    m = (mask > 0).astype(np.uint8)\n    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    if not cnts:\n        return 0.0, 0.0\n    c = max(cnts, key=cv2.contourArea)\n    (_, _), (w, h), _ = cv2.minAreaRect(c)\n    elong = max(w, h) / max(1.0, min(w, h))\n    return float(m.mean()), float(elong)\n\n\ndef build_ref(work_dir, model_dir):\n    work, out = Path(work_dir), Path(model_dir)\n    f = pd.read_csv(work / \"features.csv\", keep_default_na=False, na_values=[\"\"])\n    X = f[COLOR_FEATURE_NAMES].values.astype(np.float64)\n    from sklearn.covariance import LedoitWolf\n    mu, sd = X.mean(0), X.std(0) + 1e-6\n    Z = (X - mu) / sd\n    lw = LedoitWolf().fit(Z)\n    d = np.sqrt(np.einsum(\"ij,jk,ik->i\", Z - lw.location_, lw.precision_, Z - lw.location_))\n    gt = f[f.roi_mode == \"gt\"] if \"roi_mode\" in f.columns else f.iloc[0:0]\n    if \"mask_area_frac\" in f.columns:\n        areas, elongs = gt.mask_area_frac.dropna().tolist(), gt.mask_elong.dropna().tolist()\n    else:\n        areas, elongs = [], []\n        for _, r in gt.iterrows():\n            try:\n                a, e = mask_shape(read_mask(r.roi_mask))\n            except Exception:  # noqa: BLE001\n                continue\n            if a > 0:\n                areas.append(a)\n                elongs.append(e)\n    ref = {\n        \"feat_mu\": mu.tolist(), \"feat_sd\": sd.tolist(), \"lw_loc\": lw.location_.tolist(),\n        \"lw_prec\": lw.precision_.tolist(), \"mahal_max\": float(np.percentile(d, 99) * 1.25),\n        \"a_mean_min\": float(np.percentile(f.a_mean, 1)),\n        \"area_min\": float(np.percentile(areas, 1) * 0.5) if areas else 0.003,\n        \"area_max\": float(np.percentile(areas, 99) * 1.5) if areas else 0.35,\n        \"elong_min\": float(max(1.5, np.percentile(elongs, 2) * 0.8)) if elongs else 1.8,\n        \"n_gt_masks\": len(areas),\n    }\n    (out / \"roi_ref.json\").write_text(json.dumps(ref))\n    print(\"roi_ref.json tersimpan:\", {k: (round(v, 4) if isinstance(v, float) else v)\n                                      for k, v in ref.items() if not isinstance(v, list)})\n    return ref\n\n\ndef mahalanobis(ref, x):\n    z = (np.asarray(x, dtype=np.float64).ravel() - np.array(ref[\"feat_mu\"])) / np.array(ref[\"feat_sd\"])\n    v = z - np.array(ref[\"lw_loc\"])\n    return float(np.sqrt(v @ np.array(ref[\"lw_prec\"]) @ v))\n\n\ndef check(ref, mask, mode, feats):\n    \"\"\"Kembalikan daftar alasan penolakan (kosong = ROI layak).\"\"\"\n    if ref is None:\n        return []\n    reasons = []\n    if mode == \"pred\" and mask is not None and mask.any():\n        a, e = mask_shape(mask)\n        # batas bawah luas dilonggarkan: segmentasi v2 menangani foto HP satu mata utuh, di mana\n        # konjungtiva hanya ~0,5-2% bingkai (data latih close-up >1%). Bentuk lonjong tetap diwajibkan.\n        if not (min(ref[\"area_min\"], AREA_MIN_ABS) <= a <= ref[\"area_max\"]) or e < ref[\"elong_min\"]:\n            reasons.append(\"bentuk_roi_tidak_wajar\")\n    fdict = dict(zip(COLOR_FEATURE_NAMES, np.asarray(feats).ravel()))\n    if fdict[\"a_mean\"] < ref[\"a_mean_min\"]:\n        reasons.append(\"roi_tidak_kemerahan\")\n    if mahalanobis(ref, feats) > ref[\"mahal_max\"]:\n        reasons.append(\"warna_di_luar_data_latih\")\n    return reasons\n\n\ndef refine_user_mask(rgb, mask):\n    \"\"\"Mask dari lingkaran (lasso) pengguna. Area dipercaya apa adanya - TIDAK dipilih berdasarkan\n    kemerahan, karena konjungtiva anemia justru pucat. Hanya buang piksel yang pasti bukan jaringan:\n    bulu mata/pupil (sangat gelap) dan pantulan cahaya (putih menyilaukan).\"\"\"\n    m = (mask > 0).astype(np.uint8)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    v, s = hsv[:, :, 2], hsv[:, :, 1]\n    bad = (v < 45) | ((v > 245) & (s < 30))\n    keep = (m & ~bad).astype(np.uint8)\n    keep = cv2.morphologyEx(keep, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    return keep if keep.sum() >= 0.3 * m.sum() else m\n\n\ndef manual_mask(rgb, top_frac=0.25):\n    \"\"\"Foto sudah di-crop pengguna ke sekitar konjungtiva. Konjungtiva = jaringan paling\n    merah DAN jenuh di crop itu (kulit sawo matang juga punya a* tinggi, jadi dipakai\n    persentil atas + saturasi, bukan ambang tetap). Ambil komponen terbesar lalu rapikan.\"\"\"\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)\n    a = cv2.GaussianBlur(lab[:, :, 1], (5, 5), 0)\n    sat = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)\n    valid = rgb.sum(axis=2) > 15\n    score = (a - a[valid].mean()) / (a[valid].std() + 1e-6) + 0.5 * (sat - sat[valid].mean()) / (sat[valid].std() + 1e-6)\n    thr = np.percentile(score[valid], 100 * (1 - top_frac))\n    m = ((score >= thr) & valid).astype(np.uint8)\n    k = max(3, int(min(rgb.shape[:2]) * 0.03) | 1)\n    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((k, k), np.uint8))\n    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((k * 2 + 1, k * 2 + 1), np.uint8))\n    n, lab_cc, st, _ = cv2.connectedComponentsWithStats(m, 8)\n    if n <= 1:\n        return m\n    kk = 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])\n    return (lab_cc == kk).astype(np.uint8)\n\n\nif __name__ == \"__main__\":\n    build_ref(sys.argv[1] if len(sys.argv) > 1 else \"/content/work\",\n              sys.argv[2] if len(sys.argv) > 2 else \"/content/work/model\")\n", "app.py": "\"\"\"Langkah 10: demo web AnemiaLens (Gradio) dengan link publik sementara dari Colab.\n\nPakai:  python anemialens/app.py /content/work/model\n\"\"\"\nimport sys\nfrom pathlib import Path\n\nimport gradio as gr\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom anemialens.predict import AnemiaLens  # noqa: E402\n\nMODEL_DIR = sys.argv[1] if len(sys.argv) > 1 else \"/content/work/model\"\nlens = AnemiaLens(MODEL_DIR)\n\nGUIDE = \"\"\"\n### Cara mengambil foto\n1. Cari cahaya terang yang merata (dekat jendela atau lampu putih). **Jangan pakai flash.**\n2. **Tarik kelopak mata bawah** dengan jari sampai bagian dalamnya yang **merah muda** terlihat jelas (seperti saat diperiksa dokter).\n3. Foto **dari dekat**: mata dan kelopak bawah memenuhi sebagian besar layar. Ketuk layar untuk fokus.\n4. Jika area hijau meleset, pakai ikon **crop** di editor untuk memotong tepat di bagian dalam kelopak, lalu centang *mode crop manual*.\n5. (Disarankan) Letakkan **kartu/kertas putih polos** di samping mata agar warna dikoreksi otomatis.\n\"\"\"\nCOLOR = {\"tinggi\": \"#B91C1C\", \"sedang\": \"#B45309\", \"rendah\": \"#15803D\"}\n\n\ndef run(img, use_card, manual):\n    if img is None:\n        return None, \"Unggah atau ambil foto terlebih dahulu.\"\n    if isinstance(img, dict):                    # keluaran ImageEditor (setelah crop)\n        img = img.get(\"composite\") if img.get(\"composite\") is not None else img.get(\"background\")\n    img = np.asarray(img)\n    if img.ndim == 3 and img.shape[2] == 4:\n        img = img[:, :, :3]\n    r = lens.analyze(img, use_card=use_card, manual_crop=manual)\n    if r[\"status\"] != \"ok\":\n        tips = \"\\n\".join(f\"- {a}\" for a in r[\"advice\"])\n        return r[\"overlay\"], f\"### Foto belum bisa dinilai\\n{tips}\\n\\n_{r['disclaimer']}_\"\n    hb = \"\"\n    if r[\"hb_estimate\"] is not None:\n        err = f\" ± {r['hb_mae']:.1f}\" if r.get(\"hb_mae\") else \"\"\n        hb = f\"\\n\\n**Perkiraan Hb:** {r['hb_estimate']:.1f}{err} g/dL (perkiraan kasar, bukan hasil lab)\"\n    per = \" · \".join(f\"{k}: {v:.2f}\" for k, v in r[\"per_model\"].items())\n    warn = \"\".join(f\"\\n- {w}\" for w in r[\"quality_warnings\"])\n    md = (f\"## <span style='color:{COLOR[r['risk_tier']]}'>Risiko {r['risk_tier'].upper()}</span>\\n\"\n          f\"{r['message']}{hb}\\n\\n\"\n          f\"Skor ensemble: {r['probability']:.2f} (per model: {per})\\n\"\n          f\"Koreksi warna kartu: {'ya' if r['card_used'] else 'tidak'} · area: {r['roi_mode']}\"\n          + (f\"\\n\\n**Catatan kualitas:**{warn}\" if warn else \"\")\n          + f\"\\n\\n---\\n_{r['disclaimer']}_\")\n    return r[\"overlay\"], md\n\n\nwith gr.Blocks(title=\"AnemiaLens\") as demo:\n    gr.Markdown(\"# AnemiaLens\\nSkrining awal anemia dari foto bagian dalam kelopak mata bawah (konjungtiva).\")\n    with gr.Row():\n        with gr.Column():\n            inp = gr.ImageEditor(label=\"Foto mata (ikon crop untuk memotong manual)\", type=\"numpy\",\n                                 sources=[\"upload\", \"webcam\"], brush=False, eraser=False)\n            card = gr.Checkbox(label=\"Ada kartu putih di dalam foto (koreksi warna)\", value=False)\n            manual = gr.Checkbox(label=\"Saya sudah crop foto tepat di bagian dalam kelopak bawah (mode crop manual)\",\n                                 value=False)\n            btn = gr.Button(\"Analisis\", variant=\"primary\")\n            gr.Markdown(GUIDE)\n        with gr.Column():\n            out_img = gr.Image(label=\"Area yang dianalisis (garis hijau)\")\n            out_md = gr.Markdown()\n    btn.click(run, [inp, card, manual], [out_img, out_md])\n\nif __name__ == \"__main__\":\n    demo.launch(share=True, debug=False)\n"}
for name, text in FILES.items():
    open(f'/content/anemialens/{name}', 'w').write(text)
print('Kode siap:', sorted(FILES))

## Langkah 4 — Ambil dataset (±3–5 menit, otomatis)
Kedua dataset sudah diatur dari link Google Drive Anda:
- **Eyes-defy-anemia** (`archive.zip`): 217 pasien dewasa India + Italia, Hb lab, mask palpebral.
- **CP-AnemiC** (`CP-AnemiC.zip`): balita Ghana, Hb lab. Foto duplikat dengan Hb berbeda (303 foto) otomatis dibuang di Langkah 6, sehingga tersisa 407.

Sel ini mencoba mengunduh lewat link. Jika gagal (misalnya link tidak dibagikan sebagai *Anyone with the link*), sel otomatis mencari file dengan nama yang sama di Google Drive Anda.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, glob, shutil, subprocess

DATASETS = {   # nama folder tujuan : (ID file Google Drive, nama file)
    'eyes_defy_anemia': ('1jBDBERO18JgmC0nemU5SyUjmrW6BtAYf', 'archive.zip'),
    'cp_anemic':        ('1YUFN6rN9JxIpphffkBYicZmM3pEBPA2b', 'CP-AnemiC.zip'),
}
os.makedirs('/content/data', exist_ok=True)
for folder, (fid, fname) in DATASETS.items():
    dst = f'/content/{fname}'
    if not os.path.exists(dst):
        subprocess.run(['gdown', f'https://drive.google.com/uc?id={fid}', '-O', dst])
    if not os.path.exists(dst) or os.path.getsize(dst) < 1_000_000:
        found = glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True)
        if not found:
            raise FileNotFoundError(f'{fname} tidak bisa diunduh dan tidak ditemukan di Drive. '
                                    f'Buka link file -> Share -> Anyone with the link, lalu jalankan ulang sel ini.')
        shutil.copy(found[0], dst)
    zipfile.ZipFile(dst).extractall(f'/content/data/{folder}')
    print(f'OK {fname} -> /content/data/{folder} ({os.path.getsize(dst) / 1e6:.0f} MB)')
for _ in range(3):                                   # ekstrak zip bersarang (jika ada)
    inner = glob.glob('/content/data/**/*.zip', recursive=True)
    if not inner: break
    for z in inner:
        zipfile.ZipFile(z).extractall(os.path.splitext(z)[0]); os.remove(z)
print(sorted(os.listdir('/content/data')))

## Langkah 5 — Periksa isi dataset
Keluaran sel ini menunjukkan struktur folder, jumlah foto/mask, dan kolom tabel label (*Italy.xlsx*, *India.xlsx*, *Anemia_Data_Collection_Sheet.xlsx*).

In [ ]:
!python /content/anemialens/inspect_data.py /content/data

## Langkah 6 — Atur konfigurasi & buat manifest
`manifest.csv` menyatukan semua sumber data: path foto, mask, ID pasien, Hb, jenis kelamin, usia, dan label.
- Label diambil dari **Hb laboratorium** dengan ambang WHO (balita < 11, perempuan < 12, laki-laki < 13 g/dL). Jika tidak ada Hb, dipakai label tabel atau nama folder.
- Gambar duplikat antar sumber dibuang, supaya tidak bocor antara data latih dan data uji.

**Hasil yang benar:** total **624 foto berlabel** → `cp_anemic` 407, `eyes_defy_anemia/India` 95, `eyes_defy_anemia/Italy` 122, dan semuanya bersumber `hb_lab`.

In [ ]:
import os, sys; sys.path.insert(0, '/content')
from anemialens.common import Config
cfg = Config(
    work_dir='/content/work',
    cnn_backbone='efficientnet_b0',   # coba 'convnext_tiny.fb_in22k_ft_in1k' jika ingin lebih kuat (lebih lambat)
    cnn_epochs=20, seg_epochs=40, img_size=224,
    target_sensitivity=0.85,          # skrining: utamakan tidak melewatkan kasus anemia
)
os.makedirs(cfg.work_dir, exist_ok=True); cfg.save(f'{cfg.work_dir}/config.json')
!python /content/anemialens/build_manifest.py /content/data /content/work

## Langkah 7 — Latih segmentasi konjungtiva (±5–10 menit)
U-Net (encoder ResNet-34 ImageNet) dilatih dari mask *palpebral* yang ada di dataset dengan augmentasi **robust** (simulasi konjungtiva pucat dan foto HP: zoom-out, blur, JPEG, white balance), lalu dipakai untuk foto yang tidak punya mask. Butuh ±15 menit di T4.
- Target **Dice validasi > 0,80**.
- Sumber yang fotonya sudah berupa potongan konjungtiva (misalnya CP-AnemiC) otomatis dipakai utuh. Kalau perlu dipaksa, tambahkan `--roi-sources=NAMA_SUMBER`.

In [ ]:
!python /content/anemialens/train_seg.py /content/work --robust --epochs=60

## Langkah 8 — Potong ROI, hitung fitur warna & kualitas, lalu cek visual (±2 menit)
Lihat gambar di bawah. **Pastikan garis/area hijau tepat berada di bagian dalam kelopak mata bawah yang merah muda** (pita lengkung di bawah bola mata). Kalau banyak yang meleset, hasil model tidak bisa dipercaya, jadi hentikan dan kirim gambarnya ke Claude.

In [ ]:
!python /content/anemialens/prepare_rois.py /content/work
!python /content/anemialens/show_samples.py /content/work
from IPython.display import Image, display
for f in ['samples_mask.png', 'samples_roi.png']:
    if os.path.exists(f'/content/work/{f}'): display(Image(f'/content/work/{f}'))

## Langkah 9 — Validasi silang 5-fold per pasien

**9a. Uji cepat tanpa CNN (±2 menit).** Fitur warna + DINOv2 saja. Berguna untuk melihat apakah data dan label sudah benar.

In [ ]:
!python /content/anemialens/train_eval.py /content/work --no-cnn

**9b. Validasi lengkap dengan CNN (±20–30 menit di T4).** Hasilnya: tabel AUC, sensitivitas, spesifisitas, MAE Hb per model dan per sumber, plus uji *leave-one-source-out*.
Model dengan AUC jauh di bawah yang terbaik otomatis tidak dimasukkan ke ensemble.

In [ ]:
!python /content/anemialens/train_eval.py /content/work
from IPython.display import Image, display, Markdown
display(Markdown(open('/content/work/report.md').read()))
for f in ['roc.png', 'hb_scatter.png']:
    if os.path.exists(f'/content/work/{f}'): display(Image(f'/content/work/{f}', width=420))

**Cara membaca hasil**

| Metrik | Arti | Target awal |
|---|---|---|
| AUC | Kemampuan membedakan anemia dan normal (0,5 = menebak, 1 = sempurna) | ≥ 0,80 |
| Sensitivitas | % kasus anemia yang tertangkap | ≥ 85% (sudah diatur lewat ambang) |
| Spesifisitas | % orang normal yang tidak salah dirujuk | setinggi mungkin |
| MAE Hb | Rata-rata selisih perkiraan Hb dengan Hb lab | < 1,5 g/dL |
| Leave-one-source-out | AUC saat diuji di sumber yang tidak dilihat saat latihan | proksi validasi eksternal |

## Langkah 10 — Latih model final & unduh ke komputer (±5 menit)
Model final dilatih memakai semua data. Ambang skrining diambil dari validasi silang.
Semua artefak dikemas jadi **`AnemiaLens_model.zip`** lalu **otomatis terunduh lewat browser** ke folder *Downloads* Anda. Tidak ada yang disimpan ke Google Drive.

Setelah terunduh, pindahkan atau ekstrak zip itu ke `F:\Lomba Binus\JOINTS 2026\AnemiaLens\model\` (atau minta Claude memindahkannya).
Kalau browser memblokir unduhan, klik ikon folder di kiri Colab, buka `/content`, klik kanan `AnemiaLens_model.zip`, lalu pilih **Download**.

In [ ]:
!python /content/anemialens/train_final.py /content/work
import shutil, os
PKG = '/content/AnemiaLens_model'
shutil.rmtree(PKG, ignore_errors=True)
shutil.copytree('/content/work/model', f'{PKG}/model')
for f in ['report.md', 'oof.csv', 'manifest.csv', 'features.csv', 'roc.png', 'hb_scatter.png',
          'samples_roi.png', 'samples_mask.png']:
    if os.path.exists(f'/content/work/{f}'):
        shutil.copy(f'/content/work/{f}', f'{PKG}/{f}')
shutil.copytree('/content/anemialens', f'{PKG}/anemialens', ignore=shutil.ignore_patterns('__pycache__'))
zip_path = shutil.make_archive('/content/AnemiaLens_model', 'zip', PKG)
print(f'Siap: {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)')
from google.colab import files
files.download(zip_path)

## Langkah 11 — Coba prediksi satu foto

In [ ]:
import json, random, pandas as pd
from anemialens.predict import AnemiaLens
from anemialens.common import read_rgb
lens = AnemiaLens('/content/work/model')
row = pd.read_csv('/content/work/manifest.csv').sample(1, random_state=random.randint(0, 999)).iloc[0]
r = lens.analyze(read_rgb(row.image_path), use_card=False)
ov = r.pop('overlay')
print('Label asli:', 'ANEMIA' if row.label == 1 else 'normal', '| Hb lab:', row.hb)
print(json.dumps({k: v for k, v in r.items() if k != 'quality'}, indent=2, ensure_ascii=False, default=str))
from PIL import Image as PILImage; display(PILImage.fromarray(ov).resize((480, int(480 * ov.shape[0] / ov.shape[1]))))

## Langkah 12 — Demo web untuk juri (Gradio)
Sel ini menampilkan **link publik `*.gradio.live`** yang bisa dibuka dari HP (aktif selama Colab berjalan, maksimal sekitar 72 jam).
Unggah foto kelopak mata bawah, centang *kartu putih* bila memakai kartu referensi, lalu klik **Analisis**.
Hentikan sel untuk mematikan demo.

In [ ]:
!python /content/anemialens/app.py /content/work/model

---
### Kalau kembali lagi besok (Colab sudah reset)
Jalankan Langkah 1–3, lalu sel di bawah. Pilih file `AnemiaLens_model.zip` dari komputer Anda (mis. dari folder `JOINTS 2026\AnemiaLens`), lalu langsung lanjut ke Langkah 11/12.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                       # pilih AnemiaLens_model.zip
name = next(iter(up))
zipfile.ZipFile(name).extractall('/content/AnemiaLens_model')
os.makedirs('/content/work', exist_ok=True)
!rm -rf /content/work/model && cp -r /content/AnemiaLens_model/model /content/work/model
print('Model siap di /content/work/model')